# Mission 3 ver20 — 더 큰 한국어 문맥 모델 · 실행 예산 3시간

**이 노트북 하나와 기존 `mission3_ver5_inputs.zip`으로 실행합니다.**
Colab에서 GPU 런타임을 선택하고 **런타임 → 모두 실행**을 누르세요.
Drive 연결 요청에는 입력 ZIP이 있는 **공용 계정**을 선택합니다.

이번에는 **KLUE-RoBERTa-large의 뒤 8개 층과 9개 증상 분류기를 학습**합니다.
앞 16개 층과 임베딩은 고정해 학습 비용을 줄입니다. 기존의 단순 가중 CE로 후보 하나에 집중합니다.
기존 결과 파일을 다시 올릴 필요 없이, 같은 Training 내부 평가의 저장된 기준 점수와 비교합니다.
Large 모델의 실제 F1과 GPU 실행 시간은 아직 측정하지 않았습니다. 큰 폭의 상승을 보장하지 않습니다.
원래 정답과 모든 샘플을 유지하며, 모델 입력은 `utterances[].text`뿐입니다.
threshold 탐색·사후 보정·외부 학습 데이터 추가 없이 증상별 두 class의 `argmax`로 예측합니다.

**이 노트북 한 번의 작업 예산은 설치·데이터 준비·학습·최종 평가를 포함해 180분입니다.**
Drive 로그인 대기 시간은 제외합니다. 첫 학습에서 실제 GPU 속도를 재고 추가 epoch와 전체 Training
재학습이 가능한지 계산합니다. 느린 GPU에서는 추가 epoch 또는 전체 재학습이 생략될 수 있습니다.
완료된 모델만 최종 결과에 사용하며, 실제 학습 범위를 결과에 명시합니다.
작업 제한 시간과 모든 학습 단계의 완료는 다릅니다. 최종 완료 표시가 없는 실행은 미완료입니다.
결과 표시 후, 오류 발생 또는 시간 제한에 도달하면 GPU 런타임의 자동 반환을 요청합니다.
결과는 Drive에 남지만 `/content`의 임시 파일은 사라질 수 있습니다.

**이번에는 ver20 하나만 실행하면 됩니다. 이전 버전의 결과 파일은 필요하지 않습니다.**


## 이번에 학습하는 후보 하나

`large_partial_ce`: 공개 `klue/roberta-large`에서 새로 시작합니다.
모델의 24개 인코더 층 중 앞 16개 층과 임베딩은 고정하고, 뒤 8개 층과 분류기를 학습합니다.
입력 대화 전체를 512-token 창과 64-token 겹침으로 읽고 증상별 두 class를 예측합니다.
기존 ver15의 단순 분류 구조와 가중 cross entropy를 사용합니다.
GCE, 발화별 보조 분기, label attention, 대조학습, threshold 조정은 사용하지 않습니다.

모델은 [공식 저장소의 고정 revision](https://huggingface.co/klue/roberta-large/commit/28d911204e9022eda172571ca8cc61eaffd942f7)을 사용합니다.
뒤 8개 층 학습은 시간과 모델 적응력 사이의 실험 설정이며 최적값으로 입증된 것은 아닙니다.
원래 정답과 모든 발화·샘플을 유지하며, 이전에 학습된 모델을 가져오지 않습니다.

양성 학습 가중치는 ver15의 `recall_moderate`와 같습니다.
`[고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란]` 순서로
`[1.4, 1.8, 2.0, 1.2, 1.3, 1.0, 3.0, 1.5, 1.3]`을 사용합니다.
최대 3epoch와 3epoch 기준 학습률 감소를 사용합니다. 첫 1회 갱신으로 준비한 뒤 10회 갱신의 속도를 측정합니다.
준비와 측정 과정도 첫 epoch의 실제 학습에 포함되며, 같은 갱신을 다시 수행하지 않습니다.
기존 기준 모델을 다시 학습하지 않고 이번 후보 하나만 실행합니다. 이후 epoch는 전체 재학습 시간을 함께 고려합니다.
Training 내부에서 ver19보다 전체 F1과 오심 F1이 모두 높은 epoch 중 전체 F1이 가장 높은 것을 선택합니다.
그런 epoch가 없으면 전체 F1이 가장 높은 결과를 진단용으로 저장합니다.
동시 개선을 통과하고 시간이 허용되면 전체 Training으로 새로 학습합니다.
기존 Training 데이터·정답·분할의 해시가 맞을 때만 내장된 ver19 기준과 비교합니다.
전체 F1과 오심 F1이 **모두** 높아졌는지, 오심 누락·오탐이 어떻게 바뀌었는지 별도로 표시합니다.
동시 개선 조건을 충족하지 못하면 전체 재학습을 생략해 시간을 절약합니다.
이때 내부 fit 학습 결과와 실험용 예측은 보관하지만 새 추천 제출 모델로 표시하지 않습니다.
이전 제출 모델을 자동으로 교체하지 않습니다.
반복 사용한 Training holdout 비교는 탐색 결과이며 새 독립 평가셋의 검증으로 해석하면 안 됩니다.
공식 Validation은 최종 모델을 정한 뒤 한 번 평가하며 학습·epoch 선택에는 사용하지 않습니다.


In [ ]:
# @title 실행 설정
SETTINGS = {'version': 'ver20', 'backend': 'neural', 'model_id': 'klue/roberta-large', 'revision': '28d911204e9022eda172571ca8cc61eaffd942f7', 'trainable_encoder_layers': 8, 'expected_encoder_layers': 24, 'label_attention': False, 'contrastive_weight': 0.0, 'ce_views': 1, 'input_zip': '/content/drive/MyDrive/mission3_ver5_inputs.zip', 'output_dir': '/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver20', 'work_dir': '/content/mission3_ver20', 'train_dir': '', 'validation_dir': '', 'run_validation': True, 'seed': 42, 'holdout_size': 0.2, 'max_length': 512, 'overlap': 64, 'batch_size': 4, 'encoder_window_batch': 4, 'gradient_accumulation': 4, 'max_epochs': 3, 'scheduler_horizon_epochs': 3, 'patience': 2, 'learning_rate': 2e-05, 'head_learning_rate': 0.0001, 'refit_full_training': True, 'allow_cpu': False, 'log_every': 250, 'time_budget_minutes': 180, 'finalization_reserve_seconds': 1500, 'budget_safety_factor': 1.25, 'pilot_optimizer_steps': 10, 'resume_run': ''}
AUTO_RELEASE_GPU = True # @param {type:"boolean"}
# Keep train_dir and validation_dir empty to use the existing input ZIP.
# Resume only a compatible run; set resume_run to its complete Drive directory.
# Each explicit rerun starts a NEW time budget and uses additional Colab resources.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 3시간 작업 예산 시작 (이후 패키지 설치 시간도 포함)
import time
import threading
BUDGET_STARTED_MONOTONIC = time.monotonic()
DEADLINE_MONOTONIC = BUDGET_STARTED_MONOTONIC + float(SETTINGS["time_budget_minutes"]) * 60
SETTINGS["deadline_monotonic"] = DEADLINE_MONOTONIC
previous_timer = globals().get("RELEASE_TIMER")
if previous_timer is not None:
    previous_timer.cancel()
RELEASE_TIMER = None
RELEASE_LOCK = threading.Lock()
RELEASE_REQUESTED = False

def release_gpu():
    global RELEASE_REQUESTED
    if not AUTO_RELEASE_GPU:
        return
    with RELEASE_LOCK:
        if RELEASE_REQUESTED:
            return
        RELEASE_REQUESTED = True
        if RELEASE_TIMER is not None:
            RELEASE_TIMER.cancel()
    try:
        from google.colab import runtime as colab_runtime
        print("Colab GPU 런타임 반환을 요청합니다. 저장된 결과는 Drive에서 확인하세요.", flush=True)
        colab_runtime.unassign()
    except Exception as error:
        print("자동 반환 요청 실패:", type(error).__name__,
              "· Colab 메뉴 '런타임 → 연결 해제 및 런타임 삭제'를 직접 실행하세요.", flush=True)

if AUTO_RELEASE_GPU:
    RELEASE_TIMER = threading.Timer(max(0, DEADLINE_MONOTONIC - time.monotonic()), release_gpu)
    RELEASE_TIMER.daemon = True
    RELEASE_TIMER.start()
print("작업 시간 예산:", SETTINGS["time_budget_minutes"], "분")


In [ ]:
# @title 실행 코드 준비 (추가 파일 업로드 불필요)
import importlib.util
import sys
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Standalone Mission 3 partial fine-tuning experiment for KLUE-RoBERTa-large."""\n', 'data.py': '"""Text-only Mission 3 data, grouped Training holdout, and discrete predictions."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n        if text:\n            texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _greedy_groups(features, sizes, holdout_size, seed):\n    """Dependency-free iterative stratification of indivisible transcript groups.\n\n    This balances label presence and absence, then row counts. Group constraints\n    mean neither this fallback nor iterstrat can guarantee exact proportions.\n    """\n    rng = np.random.default_rng(seed)\n    active = np.ones(len(features), dtype=bool)\n    assignment = np.full(len(features), -1, dtype=np.int8)\n    remaining = features.sum(axis=0).astype(float)\n    proportions = np.asarray([1.0 - holdout_size, holdout_size])\n    desired_labels = proportions[:, None] * remaining[None, :]\n    desired_rows = proportions * sizes.sum()\n    coverage = np.zeros((2, features.shape[1]), dtype=bool)\n    while active.any():\n        available = np.flatnonzero(remaining > 0)\n        if not len(available):\n            candidates = np.flatnonzero(active)\n            rng.shuffle(candidates)\n            label = None\n        else:\n            rare = available[remaining[available] == remaining[available].min()]\n            label = int(rng.choice(rare))\n            candidates = np.flatnonzero(active & features[:, label].astype(bool))\n            rng.shuffle(candidates)\n        for group in candidates:\n            deficits = desired_rows if label is None else desired_labels[:, label]\n            sides = np.flatnonzero(deficits == deficits.max())\n            if len(sides) > 1:\n                sides = sides[desired_rows[sides] == desired_rows[sides].max()]\n            present = np.flatnonzero(features[group])\n            final_occurrences = present[remaining[present] == 1]\n            required = np.unique(np.where(~coverage[:, final_occurrences])[0])\n            if len(required) == 1:\n                # Reserve the last group for a side still missing this target.\n                sides = required\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            desired_rows[side] -= sizes[group]\n            desired_labels[side] -= features[group]\n            remaining -= features[group]\n            coverage[side] |= features[group].astype(bool)\n            active[group] = False\n    return np.flatnonzero(assignment == 0), np.flatnonzero(assignment == 1)\n\n\ndef make_split(texts, targets, holdout_size: float = .2, seed: int = 42):\n    """Split Training only; normalized duplicate transcripts cannot cross sides."""\n    if not 0 < holdout_size < .5:\n        raise ValueError("holdout_size must be in (0, .5)")\n    targets = _binary(targets, len(texts), "targets")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    presence = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    absence = np.zeros_like(presence)\n    np.maximum.at(presence, inverse, targets)\n    np.maximum.at(absence, inverse, 1 - targets)\n    features = np.concatenate([presence, absence], axis=1)\n    if len(unique) < 2 or np.any(features.sum(axis=0) < 2):\n        raise ValueError("Each symptom needs positive and negative examples in at least two text groups")\n    try:\n        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit\n    except ImportError:\n        MultilabelStratifiedShuffleSplit = None\n    best = None\n    # Retry only group allocation, never inspecting official Validation.\n    for attempt in range(8):\n        current_seed = seed + attempt\n        if MultilabelStratifiedShuffleSplit is None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n        else:\n            splitter = MultilabelStratifiedShuffleSplit(\n                n_splits=1, test_size=holdout_size, random_state=current_seed)\n            fit_groups, holdout_groups = next(splitter.split(np.arange(len(unique)), features))\n        fit = np.flatnonzero(np.isin(inverse, fit_groups))\n        holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n        valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                    and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if not valid and MultilabelStratifiedShuffleSplit is not None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n            fit = np.flatnonzero(np.isin(inverse, fit_groups))\n            holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n            valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                        and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if valid:\n            ratio = targets[holdout].sum(axis=0) / targets.sum(axis=0)\n            discrepancy = float(np.abs(ratio - holdout_size).mean()\n                                + abs(len(holdout) / len(targets) - holdout_size))\n            if best is None or discrepancy < best[0]:\n                best = (discrepancy, fit, holdout)\n    if best is None:\n        raise ValueError("Cannot split text groups with positive and negative examples for every symptom")\n    return best[1], best[2]\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\ndef export_errors(paths, texts, targets, predictions, output_dir, max_per_label: int = 20):\n    """Save full text locally for manual review; never print sensitive dialogue."""\n    targets = _binary(targets, len(texts), "targets")\n    predictions = _binary(predictions, len(texts))\n    if len(paths) != len(texts) or type(max_per_label) is not int or max_per_label < 1:\n        raise ValueError("paths/texts must align and max_per_label must be a positive integer")\n    output_dir = Path(output_dir)\n    outputs = []\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    header = ["label file name", "symptom", "error_type", "true_symptoms", "predicted_symptoms", "text",\n              "review_category", "review_note", "has_nausea_expression", "has_question_expression",\n              "has_negation_expression"]\n\n    def review_row(index, symptom, category):\n        actual = [name for name, flag in zip(TARGET_SYMPTOMS, targets[index]) if flag]\n        predicted = [name for name, flag in zip(TARGET_SYMPTOMS, predictions[index]) if flag]\n        return [Path(paths[index]).name, symptom, category, str(actual), str(predicted), texts[index], "", "",\n                int(signals["nausea"][index]), int(signals["question"][index]), int(signals["negation"][index])]\n\n    for label, name in enumerate(TARGET_SYMPTOMS):\n        fn = np.flatnonzero((targets[:, label] == 1) & (predictions[:, label] == 0)).tolist()\n        fp = np.flatnonzero((targets[:, label] == 0) & (predictions[:, label] == 1)).tolist()\n        indices = []\n        while (fn or fp) and len(indices) < max_per_label:\n            for pending in (fn, fp):\n                if pending and len(indices) < max_per_label:\n                    indices.append(pending.pop(0))\n        output = output_dir / f"errors_{name}.csv"\n\n        def write(handle):\n            writer = csv.writer(handle)\n            writer.writerow(header)\n            for index in indices:\n                writer.writerow(review_row(index, name, "FN" if targets[index, label] else "FP"))\n        _atomic_text(output, write)\n        outputs.append(output)\n\n    nausea_index, vomiting_index = TARGET_SYMPTOMS.index("오심"), TARGET_SYMPTOMS.index("구토")\n    nausea, vomiting = targets[:, nausea_index].astype(bool), targets[:, vomiting_index].astype(bool)\n    combinations = {"neither": ~nausea & ~vomiting, "nausea_only": nausea & ~vomiting,\n                    "vomiting_only": ~nausea & vomiting, "both": nausea & vomiting}\n    group_counts = {}\n    for name, mask in combinations.items():\n        row = {"n": int(mask.sum())}\n        for symptom, column in (("nausea", nausea_index), ("vomiting", vomiting_index)):\n            actual, predicted = targets[:, column].astype(bool), predictions[:, column].astype(bool)\n            row[f"predicted_{symptom}"] = int((mask & predicted).sum())\n            row[f"{symptom}_tp"] = int((mask & actual & predicted).sum())\n            row[f"{symptom}_fp"] = int((mask & ~actual & predicted).sum())\n            row[f"{symptom}_fn"] = int((mask & actual & ~predicted).sum())\n            row[f"{symptom}_tn"] = int((mask & ~actual & ~predicted).sum())\n        group_counts[name] = row\n    summary_path = output_dir / "nausea_vomiting_error_groups.json"\n    write_json(summary_path, group_counts)\n    outputs.append(summary_path)\n\n    predicted_nausea = predictions[:, nausea_index].astype(bool)\n    controls = {"TP": nausea & predicted_nausea, "FP": ~nausea & predicted_nausea,\n                "FN": nausea & ~predicted_nausea, "TN": ~nausea & ~predicted_nausea}\n    rng = np.random.default_rng(42)\n    sampled = []\n    for category, mask in controls.items():\n        candidates = np.flatnonzero(mask)\n        chosen = rng.choice(candidates, size=min(10, len(candidates)), replace=False)\n        sampled.extend((int(index), category) for index in chosen)\n    # Mix categories for human review while preserving reproducibility.\n    rng.shuffle(sampled)\n    review_path = output_dir / "review_오심.csv"\n\n    def write_controls(handle):\n        writer = csv.writer(handle)\n        writer.writerow(header)\n        for index, category in sampled:\n            writer.writerow(review_row(index, "오심", category))\n    _atomic_text(review_path, write_controls)\n    outputs.append(review_path)\n    return outputs\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'neural.py': '"""Text-only dialogue classifiers with ordinary binary class decisions.\n\nEach symptom has two class logits. Predictions use the winning class directly;\nthere is no probability cutoff search, calibration, or post-hoc label weighting.\nPublic pretrained assets are prepared separately from local checkpoint weights.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nfrom pathlib import Path\nimport random\nimport re\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset\n\n\nNUM_LABELS = 9\nCONTRASTIVE_DIM = 64\n_MODEL_INPUTS = ("input_ids", "attention_mask", "token_type_ids")\n\n\ndef seed_everything(seed):\n    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef configure_partial_finetuning(encoder: nn.Module, trainable_layers: int = 8,\n                                 expected_layers: int = 24) -> dict:\n    """Freeze a RoBERTa encoder except its final transformer layers.\n\n    Frozen layers also run without dropout. The classifier reapplies that mode\n    after every ``train()`` call; inference does not require this configuration.\n    All shape validation precedes changes to parameters or module modes.\n    """\n    for name, value in (("trainable_layers", trainable_layers),\n                        ("expected_layers", expected_layers)):\n        if isinstance(value, bool) or not isinstance(value, int) or value < 1:\n            raise ValueError(f"{name} must be a positive integer")\n    if trainable_layers > expected_layers:\n        raise ValueError("trainable_layers must not exceed expected_layers")\n    config = getattr(encoder, "config", None)\n    if getattr(config, "model_type", None) != "roberta":\n        raise ValueError("Partial finetuning requires a RoBERTa encoder")\n    embeddings = getattr(encoder, "embeddings", None)\n    stack = getattr(encoder, "encoder", None)\n    layers = getattr(stack, "layer", None)\n    pooler = getattr(encoder, "pooler", None)\n    if (not isinstance(embeddings, nn.Module)\n            or not isinstance(layers, nn.ModuleList)\n            or len(layers) != expected_layers\n            or getattr(config, "num_hidden_layers", None) != expected_layers\n            or any(not isinstance(layer, nn.Module) for layer in layers)\n            or (pooler is not None and not isinstance(pooler, nn.Module))):\n        raise ValueError(f"Expected RoBERTa embeddings and exactly {expected_layers} encoder layers")\n    boundary = expected_layers - trainable_layers\n    frozen_modules = [embeddings, *layers[:boundary]]\n    if pooler is not None:\n        frozen_modules.append(pooler)\n    trainable_ids = {id(parameter) for layer in layers[boundary:]\n                     for parameter in layer.parameters()}\n    frozen_ids = {id(parameter) for module in frozen_modules\n                  for parameter in module.parameters()}\n    if not trainable_ids or any(not list(layer.parameters()) for layer in layers):\n        raise ValueError("Each encoder layer must contain parameters")\n    if trainable_ids & frozen_ids:\n        raise ValueError("Frozen and trainable RoBERTa modules must not share parameters")\n    # An unexpected encoder component must not become trainable by accident.\n    for parameter in encoder.parameters():\n        parameter.requires_grad_(id(parameter) in trainable_ids)\n        parameter.grad = None\n    total = sum(parameter.numel() for parameter in encoder.parameters())\n    trainable = sum(parameter.numel() for parameter in encoder.parameters()\n                    if parameter.requires_grad)\n    audit = {\n        "model_type": "roberta",\n        "total_layers": expected_layers,\n        "trainable_layers": trainable_layers,\n        "frozen_layers": boundary,\n        "trainable_layer_indices": list(range(boundary, expected_layers)),\n        "frozen_layer_indices": list(range(boundary)),\n        "embeddings_trainable": False,\n        "pooler_trainable": False,\n        "frozen_modules_eval": True,\n        "total_parameters": total,\n        "trainable_parameters": trainable,\n        "frozen_parameters": total - trainable,\n    }\n    encoder._mission3_partial_finetuning = dict(audit)\n    enforce_frozen_eval(encoder)\n    return audit\n\n\ndef enforce_frozen_eval(encoder: nn.Module) -> None:\n    """Keep the configured frozen prefix deterministic in training mode."""\n    audit = getattr(encoder, "_mission3_partial_finetuning", None)\n    if audit is None:\n        return\n    encoder.embeddings.eval()\n    for layer in encoder.encoder.layer[:audit["frozen_layers"]]:\n        layer.eval()\n    if getattr(encoder, "pooler", None) is not None:\n        encoder.pooler.eval()\n\n\ndef prepare_encoder(model_id, revision, asset_dir):\n    """Download an encoder once; store only its config and tokenizer assets.\n\n    The caller stores the trained encoder and head together in its checkpoint.\n    Transformers imports are lazy so CPU tests do not require that dependency.\n    """\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n\n    if not isinstance(model_id, str) or not model_id.strip():\n        raise ValueError("model_id must be a nonempty string")\n    if not isinstance(revision, str) or not revision.strip():\n        raise ValueError("revision must be explicit")\n    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=False)\n    resolved = getattr(config, "_commit_hash", None)\n    if resolved is None and re.fullmatch(r"[0-9a-f]{40}", revision):\n        resolved = revision\n    if not isinstance(resolved, str) or not re.fullmatch(r"[0-9a-f]{40}", resolved):\n        raise RuntimeError("The immutable pretrained model revision was not resolved")\n    # Resolve a mutable branch exactly once, then pin every other asset to it.\n    options = {"revision": resolved, "trust_remote_code": False}\n    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, **options)\n    if not getattr(tokenizer, "is_fast", False):\n        raise ValueError("Complete overflow windows require a fast tokenizer")\n    encoder = AutoModel.from_pretrained(model_id, config=config, **options)\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    if not getattr(encoder, "supports_gradient_checkpointing", False):\n        raise ValueError("The selected encoder must support gradient checkpointing")\n    encoder.gradient_checkpointing_enable(\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n    )\n    asset_dir = Path(asset_dir)\n    encoder_dir, tokenizer_dir = asset_dir / "encoder", asset_dir / "tokenizer"\n    encoder_dir.mkdir(parents=True, exist_ok=True)\n    tokenizer_dir.mkdir(parents=True, exist_ok=True)\n    encoder.config.save_pretrained(encoder_dir)\n    tokenizer.save_pretrained(tokenizer_dir)\n    return encoder, tokenizer, resolved\n\n\ndef _binary_targets(targets, count):\n    values = torch.as_tensor(targets)\n    if values.shape != (count, NUM_LABELS):\n        raise ValueError(f"targets must have shape ({count}, {NUM_LABELS})")\n    if not bool(torch.isfinite(values).all()) or not bool(((values == 0) | (values == 1)).all()):\n        raise ValueError("targets must contain only finite binary values")\n    return values.to(dtype=torch.long)\n\n\nclass EncodeDataset(Dataset):\n    """Retain all fast-tokenizer overflow windows, including the final tokens."""\n\n    def __init__(self, texts, targets, tokenizer, max_length=512, overlap=64):\n        texts = list(texts)\n        if any(not isinstance(text, str) for text in texts):\n            raise ValueError("Every dialogue must be a text string")\n        for name, value in (("max_length", max_length), ("overlap", overlap)):\n            if isinstance(value, bool) or not isinstance(value, int):\n                raise ValueError(f"{name} must be an integer")\n        usable = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n        if usable < 1 or not 0 <= overlap < usable:\n            raise ValueError("max_length must fit special tokens and 0 <= overlap < usable tokens")\n        if getattr(tokenizer, "pad_token_id", None) is None:\n            raise ValueError("The tokenizer must define a padding token")\n        separator = getattr(tokenizer, "sep_token", None)\n        if not isinstance(separator, str) or not separator:\n            raise ValueError("The tokenizer must define an utterance separator token")\n        self.tokenizer = tokenizer\n        self.targets = None if targets is None else _binary_targets(targets, len(texts)).clone()\n        self.documents = [[] for _ in texts]\n        for start in range(0, len(texts), 128):\n            dialogues = [re.sub(r"\\r\\n?|\\n", f" {separator} ", text)\n                         for text in texts[start:start + 128]]\n            encoded = tokenizer(\n                dialogues, truncation=True, max_length=max_length, stride=overlap,\n                return_overflowing_tokens=True, padding=False,\n                return_attention_mask=True,\n            )\n            if "overflow_to_sample_mapping" not in encoded:\n                raise ValueError("A fast tokenizer must return overflow_to_sample_mapping")\n            if "input_ids" not in encoded or "attention_mask" not in encoded:\n                raise ValueError("The tokenizer must return input_ids and attention_mask")\n            for index, source in enumerate(encoded["overflow_to_sample_mapping"]):\n                source = int(source)\n                if not 0 <= source < len(dialogues):\n                    raise ValueError("Tokenizer overflow mapping is out of range")\n                window = {key: list(encoded[key][index])\n                          for key in _MODEL_INPUTS if key in encoded}\n                length = len(window["input_ids"])\n                if not 0 < length <= max_length or any(len(value) != length for value in window.values()):\n                    raise ValueError("Tokenizer returned an invalid window length")\n                if not all(value in (0, 1) for value in window["attention_mask"]) or not any(window["attention_mask"]):\n                    raise ValueError("A real token window must have a binary, nonempty attention mask")\n                self.documents[start + source].append(window)\n        if any(not document for document in self.documents):\n            raise ValueError("Tokenizer returned a dialogue without any windows")\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["targets"] = self.targets[index]\n        return item\n\n    def collate(self, examples):\n        """Pad tokens and windows; keep one target vector for each dialogue."""\n        if not examples or any(not item["windows"] for item in examples):\n            raise ValueError("A batch must contain nonempty dialogue windows")\n        keys = set(examples[0]["windows"][0])\n        if any(set(window) != keys for item in examples for window in item["windows"]):\n            raise ValueError("Token windows must use the same model input fields")\n        if any(("targets" in item) != ("targets" in examples[0]) for item in examples):\n            raise ValueError("A batch cannot mix labeled and unlabeled dialogues")\n        batch_size = len(examples)\n        window_count = max(len(item["windows"]) for item in examples)\n        token_count = max(len(window["input_ids"]) for item in examples for window in item["windows"])\n        batch = {key: torch.full(\n            (batch_size, window_count, token_count),\n            self.tokenizer.pad_token_id if key == "input_ids" else 0,\n            dtype=torch.long,\n        ) for key in _MODEL_INPUTS if key in keys}\n        batch["window_mask"] = torch.zeros((batch_size, window_count), dtype=torch.bool)\n        for row, item in enumerate(examples):\n            for column, window in enumerate(item["windows"]):\n                length = len(window["input_ids"])\n                for key in keys:\n                    batch[key][row, column, :length] = torch.tensor(window[key], dtype=torch.long)\n                batch["window_mask"][row, column] = True\n        if "targets" in examples[0]:\n            batch["targets"] = torch.stack([item["targets"] for item in examples])\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Train nine independent binary heads from all windows of each dialogue."""\n\n    def __init__(self, encoder, label_attention=False, contrastive=False, feature_dim=256):\n        super().__init__()\n        if isinstance(feature_dim, bool) or not isinstance(feature_dim, int) or feature_dim < 1:\n            raise ValueError("feature_dim must be a positive integer")\n        if not isinstance(label_attention, bool) or not isinstance(contrastive, bool):\n            raise ValueError("label_attention and contrastive must be boolean")\n        self.encoder = encoder\n        self.label_attention = label_attention\n        self.contrastive = contrastive\n        self.encoder_window_batch = 8\n        hidden_size = int(encoder.config.hidden_size)\n        self.projection = nn.Sequential(nn.Linear(hidden_size * 2, feature_dim), nn.GELU())\n        self.dropout = nn.Dropout(0.1)\n        if label_attention:\n            self.token_queries = nn.Parameter(torch.empty(NUM_LABELS, hidden_size))\n            self.window_queries = nn.Parameter(torch.empty(NUM_LABELS, feature_dim))\n            nn.init.normal_(self.token_queries, std=0.02)\n            nn.init.normal_(self.window_queries, std=0.02)\n        else:\n            self.register_parameter("token_queries", None)\n            self.register_parameter("window_queries", None)\n        self.class_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, 2))\n        self.class_bias = nn.Parameter(torch.zeros(NUM_LABELS, 2))\n        self.feature_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, CONTRASTIVE_DIM))\n        self.feature_bias = nn.Parameter(torch.zeros(NUM_LABELS, CONTRASTIVE_DIM))\n        nn.init.xavier_uniform_(self.class_weights)\n        nn.init.xavier_uniform_(self.feature_weights)\n        enforce_frozen_eval(self.encoder)\n\n    def train(self, mode: bool = True):\n        super().train(mode)\n        if mode:\n            enforce_frozen_eval(self.encoder)\n        return self\n\n    def forward(self, batch):\n        window_mask = batch.get("window_mask")\n        ids, token_mask = batch.get("input_ids"), batch.get("attention_mask")\n        if not isinstance(ids, torch.Tensor) or ids.ndim != 3:\n            raise ValueError("input_ids must have shape (batch, windows, tokens)")\n        if not isinstance(token_mask, torch.Tensor) or token_mask.shape != ids.shape:\n            raise ValueError("attention_mask must match input_ids")\n        if not isinstance(window_mask, torch.Tensor) or window_mask.shape != ids.shape[:2] or window_mask.dtype != torch.bool:\n            raise ValueError("window_mask must be a boolean (batch, windows) tensor")\n        if ids.shape[0] < 1 or not bool(window_mask.any(dim=1).all()):\n            raise ValueError("Every dialogue must contain at least one real window")\n        if not bool(((token_mask == 0) | (token_mask == 1)).all()):\n            raise ValueError("attention_mask must be binary")\n        if not bool((token_mask.sum(-1)[window_mask] > 0).all()):\n            raise ValueError("Real windows must contain attended tokens")\n        chunk_size = self.encoder_window_batch\n        if isinstance(chunk_size, bool) or not isinstance(chunk_size, int) or chunk_size < 1:\n            raise ValueError("encoder_window_batch must be a positive integer")\n        flat_mask = window_mask.flatten()\n        inputs = {}\n        for key in _MODEL_INPUTS:\n            if key in batch:\n                if batch[key].shape != ids.shape:\n                    raise ValueError(f"{key} must match input_ids")\n                inputs[key] = batch[key].flatten(0, 1)[flat_mask]\n        representations = []\n        for start in range(0, len(inputs["input_ids"]), chunk_size):\n            chunk = {key: value[start:start + chunk_size] for key, value in inputs.items()}\n            hidden = self.encoder(**chunk).last_hidden_state\n            if not bool(torch.isfinite(hidden).all()):\n                raise FloatingPointError("Encoder produced nonfinite hidden states")\n            if self.label_attention:\n                token_scores = torch.einsum("nth,lh->ntl", hidden.float(), self.token_queries.float())\n                token_scores = token_scores / math.sqrt(hidden.shape[-1])\n                token_scores = token_scores.masked_fill(~chunk["attention_mask"].bool().unsqueeze(-1), -torch.inf)\n                token_attention = torch.softmax(token_scores, dim=1).to(hidden.dtype)\n                context = torch.einsum("ntl,nth->nlh", token_attention, hidden)\n                cls = hidden[:, 0].unsqueeze(1).expand(-1, NUM_LABELS, -1)\n                representations.append(self.projection(torch.cat((cls, context), dim=-1)))\n            else:\n                mask = chunk["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n                mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)\n                representations.append(self.projection(torch.cat((hidden[:, 0], mean), dim=-1)))\n        valid = self.dropout(torch.cat(representations, dim=0))\n        if not bool(torch.isfinite(valid).all()):\n            raise FloatingPointError("Window projection produced nonfinite features")\n        padded_shape = (*window_mask.shape, *valid.shape[1:])\n        windows = valid.new_zeros(padded_shape)\n        windows = windows.flatten(0, 1).index_copy(\n            0, flat_mask.nonzero(as_tuple=False).flatten(), valid,\n        ).reshape(padded_shape)\n        if self.label_attention:\n            scores = (windows.float() * self.window_queries.float()[None, None]).sum(-1)\n            scores = scores / math.sqrt(windows.shape[-1])\n            scores = scores.masked_fill(~window_mask.unsqueeze(-1), -torch.inf)\n            attention = torch.softmax(scores, dim=1).to(windows.dtype)\n            pooled = (attention.unsqueeze(-1) * windows).sum(dim=1)\n        else:\n            pooled = windows.sum(dim=1) / window_mask.sum(dim=1, keepdim=True).to(windows.dtype)\n            pooled = pooled.unsqueeze(1).expand(-1, NUM_LABELS, -1)\n        pooled = self.dropout(pooled)\n        logits = torch.einsum("bld,ldc->blc", pooled, self.class_weights) + self.class_bias\n        features = torch.einsum("bld,ldf->blf", pooled, self.feature_weights) + self.feature_bias\n        features = F.normalize(features.float(), dim=-1)\n        if not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(features).all()):\n            raise FloatingPointError("Classifier produced nonfinite logits or features")\n        return logits, features\n\n\ndef predictions(logits):\n    """The learned head\'s winning class; no fitted post-training cutoff."""\n    if logits.ndim != 3 or logits.shape[1:] != (NUM_LABELS, 2):\n        raise ValueError("logits must have shape (batch, 9, 2)")\n    if not bool(torch.isfinite(logits).all()):\n        raise FloatingPointError("Cannot predict from nonfinite logits")\n    return logits.argmax(dim=-1)\n\n\ndef classification_loss(logits, targets, nausea_positive_weight=1.0, *,\n                        positive_class_weights=None):\n    """Weight positive CE items by label, retaining the batch-by-label mean.\n\n    The scalar nausea weight remains supported for existing callers. Weights\n    affect supervised training only; prediction still uses the raw argmax.\n    """\n    if isinstance(nausea_positive_weight, bool) or not isinstance(nausea_positive_weight, Real):\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    try:\n        positive_weight = float(nausea_positive_weight)\n    except (OverflowError, ValueError) as error:\n        raise ValueError("nausea_positive_weight must be a finite positive real number") from error\n    if not math.isfinite(positive_weight) or positive_weight <= 0:\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    if positive_class_weights is None:\n        class_weights = [1.0] * NUM_LABELS\n        class_weights[6] = positive_weight\n    else:\n        if positive_weight != 1.0:\n            raise ValueError("Use positive_class_weights or a nondefault nausea_positive_weight, not both")\n        if not isinstance(positive_class_weights, (list, tuple)) or len(positive_class_weights) != NUM_LABELS:\n            raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n        class_weights = []\n        for value in positive_class_weights:\n            if isinstance(value, bool) or not isinstance(value, Real):\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n            try:\n                weight = float(value)\n            except (OverflowError, ValueError) as error:\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers") from error\n            if not math.isfinite(weight) or weight <= 0:\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n            class_weights.append(weight)\n    predictions(logits)\n    targets = _binary_targets(targets, len(logits)).to(logits.device)\n    if len(logits) < 1:\n        raise ValueError("classification_loss requires a nonempty batch")\n    if all(weight == 1.0 for weight in class_weights):\n        loss = F.cross_entropy(logits.float().reshape(-1, 2), targets.reshape(-1))\n    else:\n        per_item = F.cross_entropy(\n            logits.float().reshape(-1, 2), targets.reshape(-1), reduction="none",\n        ).reshape(-1, NUM_LABELS)\n        positive_weights = per_item.new_tensor(class_weights).unsqueeze(0)\n        weights = torch.where(targets == 1, positive_weights, 1.0)\n        loss = (per_item * weights).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Classification loss is nonfinite")\n    return loss\n\n\ndef binary_supervised_contrastive_loss(features1, features2, targets, temperature=0.1):\n    """Binary supervised contrastive loss per symptom, with balanced anchors.\n\n    The two views come from independent dropout forwards of the same dialogues.\n    Same-class examples are positives, self matches are excluded, and each\n    present binary class receives equal anchor weight within each symptom.\n    Thus the many symptom-negative examples cannot dominate the anchor average.\n    """\n    if features1.ndim != 3 or features1.shape[1] != NUM_LABELS or features1.shape != features2.shape:\n        raise ValueError("Both feature views must have matching (batch, 9, feature_dim) shapes")\n    if features1.shape[0] < 1 or features1.shape[2] < 1:\n        raise ValueError("Contrastive loss requires nonempty feature views")\n    if not bool(torch.isfinite(features1).all()) or not bool(torch.isfinite(features2).all()):\n        raise FloatingPointError("Contrastive features must be finite")\n    if isinstance(temperature, bool) or not math.isfinite(float(temperature)) or temperature <= 0:\n        raise ValueError("temperature must be positive and finite")\n    labels = _binary_targets(targets, len(features1)).to(features1.device)\n    features = F.normalize(torch.cat((features1.float(), features2.float()), dim=0), dim=-1)\n    labels = torch.cat((labels, labels), dim=0)\n    self_mask = torch.eye(len(features), dtype=torch.bool, device=features.device)\n    losses = []\n    for label in range(NUM_LABELS):\n        scores = features[:, label] @ features[:, label].T / float(temperature)\n        denominator = torch.logsumexp(scores.masked_fill(self_mask, -torch.inf), dim=1, keepdim=True)\n        log_probability = scores - denominator\n        positive = (labels[:, label, None] == labels[None, :, label]) & ~self_mask\n        anchor_loss = -(log_probability * positive).sum(1) / positive.sum(1)\n        class_losses = [anchor_loss[labels[:, label] == value].mean()\n                        for value in (0, 1) if bool((labels[:, label] == value).any())]\n        losses.append(torch.stack(class_losses).mean())\n    loss = torch.stack(losses).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Supervised contrastive loss is nonfinite")\n    return loss\n', 'budget.py': '"""Share a wall-clock deadline across setup, training, and final export.\n\nCreate the monotonic deadline once per notebook invocation, before package\ninstallation. Do not restore a monotonic timestamp from an earlier Colab VM.\n``Budget`` reserves finalization time when ``training=True``; subprocess\nsupervision always enforces the actual invocation deadline.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nimport os\nimport signal\nimport subprocess\nimport time\n\n\nclass BudgetExceeded(TimeoutError):\n    """Work stopped without declaring an incomplete artifact complete."""\n\n    def __init__(self, stage, *, deadline_monotonic=None):\n        self.stage = str(stage)\n        self.deadline_monotonic = deadline_monotonic\n        super().__init__(f"Wall-clock budget exhausted during {self.stage}")\n\n\ndef _finite_nonnegative(value, name):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    return value\n\n\nclass Budget:\n    """A deadline with conservative admission checks for expensive stages.\n\n    ``remaining(training=True)`` subtracts the export/evaluation reserve.\n    ``check`` raises at an exhausted deadline and otherwise returns seconds left.\n    ``can_start`` applies the safety factor to an estimated duration; it does not\n    extend the deadline or admit work when no time remains.\n    """\n\n    def __init__(self, deadline_monotonic, finalization_reserve_s=1200,\n                 safety_factor=1.25, clock=time.monotonic):\n        self.deadline_monotonic = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n        self.finalization_reserve_s = _finite_nonnegative(finalization_reserve_s, "finalization_reserve_s")\n        self.safety_factor = _finite_nonnegative(safety_factor, "safety_factor")\n        if self.safety_factor < 1:\n            raise ValueError("safety_factor must be at least one")\n        if not callable(clock):\n            raise ValueError("clock must be callable")\n        self.clock = clock\n\n    def remaining(self, training=False):\n        if not isinstance(training, bool):\n            raise ValueError("training must be a boolean")\n        now = _finite_nonnegative(self.clock(), "clock result")\n        reserve = self.finalization_reserve_s if training else 0\n        return max(0.0, self.deadline_monotonic - reserve - now)\n\n    def check(self, stage, training=False):\n        remaining = self.remaining(training=training)\n        if remaining <= 0:\n            raise BudgetExceeded(stage, deadline_monotonic=self.deadline_monotonic)\n        return remaining\n\n    def can_start(self, estimated_seconds, training=True):\n        estimate = _finite_nonnegative(estimated_seconds, "estimated_seconds")\n        remaining = self.remaining(training=training)\n        return remaining > 0 and estimate * self.safety_factor <= remaining\n\n\ndef _signal_process(process, force=False):\n    """Signal only this supervisor\'s new process group (or Windows child)."""\n    try:\n        if os.name == "posix":\n            # Popen(start_new_session=True) makes this child\'s PID its group ID.\n            # Signal the group even if its leader exited, to stop descendants.\n            os.killpg(process.pid, signal.SIGKILL if force else signal.SIGTERM)\n        elif process.poll() is None:\n            process.kill() if force else process.terminate()\n    except ProcessLookupError:\n        pass\n\n\ndef _stop_process(process, deadline_monotonic):\n    """Use at most the remaining shutdown margin for graceful stop and reaping."""\n    _signal_process(process)\n    remaining = max(0.0, deadline_monotonic - time.monotonic())\n    try:\n        process.wait(timeout=min(5.0, remaining / 2))\n    except subprocess.TimeoutExpired:\n        pass\n    # A descendant may ignore SIGTERM even when the group leader has exited.\n    _signal_process(process, force=True)\n    try:\n        process.wait(timeout=max(0.0, deadline_monotonic - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        # SIGKILL/TerminateProcess has already been issued; never wait unbounded.\n        pass\n\n\ndef run_supervised(command, deadline_monotonic, shutdown_margin_s=10, cwd=None, env=None):\n    """Run a command with inherited output and a bounded wall-clock watchdog.\n\n    Returns zero on success. Nonzero exit raises ``CalledProcessError``; timeout\n    raises ``BudgetExceeded`` after terminating the child. POSIX descendants are\n    stopped as a group; Windows supervision stops only its own direct child.\n    The shutdown margin is inside the supplied deadline, not added after it.\n    Commands must be argument sequences and are never executed through a shell.\n    """\n    deadline = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n    margin = _finite_nonnegative(shutdown_margin_s, "shutdown_margin_s")\n    if margin <= 0:\n        raise ValueError("shutdown_margin_s must be positive")\n    if isinstance(command, (str, bytes)) or not isinstance(command, (list, tuple)) or not command:\n        raise ValueError("command must be a nonempty argument sequence")\n    if deadline - margin - time.monotonic() <= 0:\n        raise BudgetExceeded("subprocess startup", deadline_monotonic=deadline)\n    process = subprocess.Popen(command, cwd=cwd, env=env, start_new_session=os.name == "posix")\n    try:\n        returncode = process.wait(timeout=max(0.0, deadline - margin - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        _stop_process(process, deadline)\n        raise BudgetExceeded("subprocess execution", deadline_monotonic=deadline) from None\n    except BaseException:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise\n    if returncode:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise subprocess.CalledProcessError(returncode, command)\n    return returncode\n', 'runtime.py': '"""Training-only, single-view symptom experiments within one shared time budget."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport time\nimport uuid\n\nimport numpy as np\n\nfrom . import data\nfrom .bootstrap import prepare_data\nfrom .budget import Budget, BudgetExceeded\n\n\nCANDIDATES = (\n    ("large_partial_ce", [1.4, 1.8, 2.0, 1.2, 1.3, 1.0, 3.0, 1.5, 1.3]),\n)\nSELECTION_POLICY = ("Training only: highest Macro F1 among epochs improving both Macro and nausea F1 "\n                    "over verified ver19; otherwise highest Macro F1 for diagnosis without full refit")\n\n\ndef _stamp():\n    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n\n\ndef _read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef _digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                     allow_nan=False).encode("utf-8")).hexdigest()\n\n\ndef dataset_digest(texts, targets):\n    digest = hashlib.sha256()\n    for text, target in zip(texts, targets):\n        digest.update(text.encode("utf-8"))\n        digest.update(b"\\0")\n        digest.update(np.asarray(target, dtype=np.int8).tobytes())\n    return digest.hexdigest()\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _safe(root, relative):\n    candidate = root / relative\n    path = candidate.resolve()\n    if not path.is_relative_to(root.resolve()) or path == root.resolve():\n        raise ValueError("Artifact path must stay inside its run directory")\n    for part in (candidate, *candidate.parents):\n        if part == root.parent:\n            break\n        if part.is_symlink():\n            raise ValueError("Symlink artifacts are not supported")\n    return path\n\n\ndef _verify_hashes(root, hashes, budget=None):\n    if not isinstance(hashes, dict) or not hashes:\n        raise ValueError("Completed artifacts require nonempty hashes")\n    for relative, expected in hashes.items():\n        if budget:\n            budget.check("verify saved artifacts")\n        path = _safe(root, relative)\n        if not path.is_file() or _hash(path) != expected:\n            raise ValueError("Completed artifact is missing or changed: " + relative)\n\n\ndef _model_hashes(root, manifest, extras=()):\n    assets = _safe(root, manifest["assets"])\n    weights = _safe(root, manifest["weights"])\n    if not weights.is_file() or not assets.is_dir():\n        raise FileNotFoundError("Model weights or assets are missing")\n    files = [weights, *sorted(p for p in assets.rglob("*") if p.is_file()),\n             *(_safe(root, name) for name in extras)]\n    if not any(p.parent.name == "tokenizer" for p in files):\n        raise FileNotFoundError("Tokenizer assets are missing")\n    return {p.relative_to(root.resolve()).as_posix(): _hash(p) for p in files}\n\n\ndef _publish_tree(source, destination, budget):\n    """Copy immutable local artifacts; caller writes completion marker last."""\n    destination.mkdir(parents=True, exist_ok=False)\n    for path in sorted(source.rglob("*")):\n        if not path.is_file():\n            continue\n        budget.check("publish completed model")\n        target = destination / path.relative_to(source)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        temporary = target.with_name(target.name + ".copying")\n        with path.open("rb") as reader, temporary.open("wb") as writer:\n            while block := reader.read(4 * 1024 * 1024):\n                budget.check("copy model to output storage")\n                writer.write(block)\n        if _hash(path) != _hash(temporary):\n            raise OSError("Published model copy failed its content hash check")\n        temporary.replace(target)\n\n\ndef validate_settings(settings):\n    result = dict(settings)\n    if result.get("version") != "ver20" or result.get("backend") != "neural":\n        raise ValueError("This runner supports ver20 neural experiments only")\n    if result.get("label_attention", False) or result.get("contrastive_weight", 0) != 0:\n        raise ValueError("ver20 uses a single CE view without attention or contrastive loss")\n    if result.get("model_id") != "klue/roberta-large" or result.get("revision") != "28d911204e9022eda172571ca8cc61eaffd942f7":\n        raise ValueError("ver20 requires the pinned KLUE RoBERTa large checkpoint")\n    if (type(result.get("expected_encoder_layers")) is not int or result["expected_encoder_layers"] != 24\n            or type(result.get("trainable_encoder_layers")) is not int or result["trainable_encoder_layers"] != 8):\n        raise ValueError("ver20 freezes embeddings and the first 16 of 24 layers, training only the last 8")\n    if result.get("loss_mode", "ce") != "ce" or result.get("utterance_evidence", False):\n        raise ValueError("ver20 requires ordinary weighted CE and the global dialogue classifier")\n    if type(result.get("pilot_optimizer_steps")) is not int or result["pilot_optimizer_steps"] < 1:\n        raise ValueError("pilot_optimizer_steps must be a positive integer")\n    if result.get("refit_full_training") is not True:\n        raise ValueError("Full refit is requested and skipped only when the time budget cannot fit it")\n    for key in ("batch_size", "encoder_window_batch", "gradient_accumulation", "max_epochs", "patience", "max_length", "log_every"):\n        if type(result.get(key)) is not int or result[key] < 1:\n            raise ValueError(key + " must be a positive integer")\n    if not 1 <= result["max_epochs"] <= 3:\n        raise ValueError("ver20 supports at most three epochs per training phase")\n    if type(result.get("scheduler_horizon_epochs")) is not int or result["scheduler_horizon_epochs"] != 3:\n        raise ValueError("ver20 retains a fixed three-epoch learning-rate schedule")\n    if type(result.get("seed")) is not int or result["seed"] < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    if type(result.get("overlap")) is not int or not 0 <= result["overlap"] < result["max_length"] - 2:\n        raise ValueError("overlap must fit within a token window")\n    if not 0 < result.get("holdout_size", 0) < .5:\n        raise ValueError("holdout_size must lie in (0, .5)")\n    for key in ("learning_rate", "head_learning_rate", "time_budget_minutes", "budget_safety_factor", "finalization_reserve_seconds"):\n        value = result.get(key)\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:\n            raise ValueError(key + " must be positive and finite")\n    if result["time_budget_minutes"] > 180:\n        raise ValueError("The execution budget cannot exceed 180 minutes")\n    if result["finalization_reserve_seconds"] >= result["time_budget_minutes"] * 60:\n        raise ValueError("Finalization reserve must be smaller than the total budget")\n    if result["budget_safety_factor"] < 1:\n        raise ValueError("budget_safety_factor must be at least one")\n    if not isinstance(result.get("resume_run", ""), str):\n        raise ValueError("resume_run must be a run-directory path or empty string")\n    result.update(label_attention=False, contrastive_weight=0.0, ce_views=1)\n    return result\n\n\ndef _stable_settings(settings):\n    excluded = {"deadline_monotonic", "input_zip", "train_dir", "validation_dir", "work_dir", "output_dir",\n                "resume_run", "log_every", "time_budget_minutes", "finalization_reserve_seconds", "budget_safety_factor"}\n    return {key: value for key, value in settings.items() if key not in excluded}\n\n\ndef _loader(texts, targets, tokenizer, settings, shuffle):\n    from torch.utils.data import DataLoader\n    from .neural import EncodeDataset\n    dataset = EncodeDataset(texts, targets, tokenizer, settings["max_length"], settings["overlap"])\n    return DataLoader(dataset, batch_size=settings["batch_size"], shuffle=shuffle,\n                      num_workers=0, collate_fn=dataset.collate)\n\n\ndef _save_weights(model, path):\n    import torch\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, temporary)\n    temporary.replace(path)\n\n\ndef _load_weights(model, path):\n    import torch\n    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)\n\n\ndef _predict(model, loader, device, budget=None, training=False):\n    import torch\n    from .neural import predictions\n    result = []\n    model.eval()\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    with torch.inference_mode():\n        for batch in loader:\n            if budget:\n                budget.check("evaluate model", training=training)\n            with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda"):\n                logits, _ = model({key: value.to(device) for key, value in batch.items()})\n            result.append(predictions(logits).cpu().numpy())\n    if not result:\n        raise ValueError("Cannot predict an empty dataset")\n    return np.concatenate(result).astype(np.int8)\n\n\ndef _optimizer(model, settings, batches, device):\n    import torch\n    identifiers = {id(p) for p in model.encoder.parameters()}\n    encoder = [p for p in model.encoder.parameters() if p.requires_grad]\n    heads = [p for p in model.parameters() if id(p) not in identifiers and p.requires_grad]\n    if not encoder or not heads:\n        raise ValueError("Both trainable encoder layers and classification heads are required")\n    optimizer = torch.optim.AdamW([\n        {"params": encoder, "lr": settings["learning_rate"]},\n        {"params": heads, "lr": settings["head_learning_rate"]}], weight_decay=.01)\n    updates = math.ceil(batches / settings["gradient_accumulation"]) * settings["scheduler_horizon_epochs"]\n    warmup = max(1, int(updates * .1))\n    def schedule(step):\n        return (step + 1) / warmup if step < warmup else max(0., (updates - step) / max(1, updates - warmup))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n    cuda = device.type == "cuda"\n    dtype = torch.bfloat16 if cuda and torch.cuda.is_bf16_supported() else torch.float16\n    scaler = torch.amp.GradScaler("cuda", enabled=cuda and dtype == torch.float16)\n    return optimizer, scheduler, scaler, dtype\n\n\ndef _pilot_projection(settings, *, microbatches, measured_batches, measured_seconds,\n                      total_batches, holdout_batches, full_ratio, budget, device, peak_memory_gb=0.):\n    """Estimate work from real first-epoch updates; never create extra updates."""\n    if measured_batches < 1 or measured_seconds <= 0:\n        raise ValueError("Throughput pilot must contain measured training batches")\n    per_batch = measured_seconds / measured_batches\n    epoch_seconds = per_batch * total_batches\n    # Using training-batch cost for inference deliberately overestimates it.\n    evaluate_seconds = per_batch * holdout_batches\n    unfinished_epoch = per_batch * max(0, total_batches - microbatches)\n    first_completion = unfinished_epoch + evaluate_seconds + 120\n    feasible = [epochs for epochs in range(1, settings["max_epochs"] + 1)\n                if budget.can_start(first_completion + (epochs - 1) * (epoch_seconds + evaluate_seconds)\n                                    + epochs * epoch_seconds * full_ratio + 120, training=True)]\n    decision = ("continue_with_refit_projection" if feasible else\n                "fit_only_risk" if budget.can_start(first_completion, training=True) else "stop_no_complete_epoch")\n    return {"observed_optimizer_steps": math.ceil(microbatches / settings["gradient_accumulation"]),\n            "observed_microbatches": microbatches, "measured_microbatches": measured_batches,\n            "measured_seconds": measured_seconds, "estimated_epoch_seconds": epoch_seconds,\n            "estimated_holdout_seconds": evaluate_seconds,\n            "remaining_seconds": budget.remaining(), "projected_feasible_epochs": max(feasible, default=0),\n            "decision": decision, "device": str(device), "peak_cuda_memory_gb": peak_memory_gb,\n            "includes_training_updates": True, "resets_or_repeats_updates": False,\n            "limitations": "Early shuffled batches estimate throughput only; later sequence lengths, I/O and GPU contention may differ."}\n\n\ndef _train_epoch(model, loader, settings, device, state, budget, epoch, pilot_context=None):\n    import torch\n    from .neural import classification_loss\n    optimizer, scheduler, scaler, dtype = state\n    accumulation = settings["gradient_accumulation"]\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    loss_total, counts = 0., np.zeros(9, dtype=np.int64)\n    started = time.monotonic()\n    measure_started, measure_start_batch, pilot_written = started, 0, False\n    pilot_batch = min(len(loader), (settings["pilot_optimizer_steps"] + 1) * accumulation)\n    if pilot_context and device.type == "cuda":\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n    for step, batch in enumerate(loader):\n        budget.check("train microbatch", training=True)\n        counts += batch["targets"].sum(0).numpy()\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda"):\n            logits, _ = model(batch)\n            loss = classification_loss(logits, batch["targets"], positive_class_weights=settings["positive_class_weights"])\n        if not bool(torch.isfinite(loss)):\n            raise FloatingPointError("Nonfinite training loss")\n        group_start = step // accumulation * accumulation\n        group_size = min(accumulation, len(loader) - group_start)\n        scaler.scale(loss / group_size).backward()\n        loss_total += float(loss.detach())\n        if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            optimizer.zero_grad(set_to_none=True)\n        if pilot_context and step + 1 == accumulation and len(loader) > accumulation:\n            if device.type == "cuda":\n                torch.cuda.synchronize()\n            measure_started, measure_start_batch = time.monotonic(), step + 1\n        if pilot_context and not pilot_written and step + 1 == pilot_batch:\n            if device.type == "cuda":\n                torch.cuda.synchronize()\n            report = _pilot_projection(\n                settings, microbatches=step + 1, measured_batches=step + 1 - measure_start_batch,\n                measured_seconds=max(time.monotonic() - measure_started, 1e-9), total_batches=len(loader),\n                holdout_batches=pilot_context["holdout_batches"], full_ratio=pilot_context["full_ratio"],\n                budget=budget, device=device,\n                peak_memory_gb=torch.cuda.max_memory_allocated() / 1024**3 if device.type == "cuda" else 0.)\n            for path in pilot_context["report_paths"]:\n                data.write_json(path, report)\n            pilot_written = True\n            print(f"Large throughput pilot: estimated epoch {report[\'estimated_epoch_seconds\'] / 60:.1f} min; "\n                  f"projected fit+refit epochs {report[\'projected_feasible_epochs\']}; {report[\'decision\']}", flush=True)\n            if report["decision"] == "stop_no_complete_epoch":\n                raise BudgetExceeded("pilot predicts no completed epoch within the training budget")\n        if (step + 1) % settings["log_every"] == 0:\n            print(f"{settings[\'candidate_id\']} epoch {epoch}: {step + 1}/{len(loader)} batches; "\n                  f"{budget.remaining() / 60:.1f} minutes left", flush=True)\n    return {"epoch": epoch, "loss": loss_total / len(loader), "classification_loss": loss_total / len(loader),\n            "ce_views": 1, "scheduler_horizon_epochs": settings["scheduler_horizon_epochs"],\n            "positive_target_counts": counts.tolist(), "training_seconds": time.monotonic() - started}\n\n\ndef _selection_key(metrics, reference_metrics=None):\n    qualifies = bool(reference_metrics is not None\n                     and metrics["macro_f1"] > reference_metrics["macro_f1"]\n                     and metrics["per_symptom"][data.TARGET_SYMPTOMS[6]]["f1"]\n                         > reference_metrics["per_symptom"][data.TARGET_SYMPTOMS[6]]["f1"])\n    return qualifies, metrics["macro_f1"]\n\n\ndef _fit_selection(model, fit_loader, hold_loader, truth, indices, settings, device, local, budget, full_ratio,\n                   pilot_report_path=None, reference_metrics=None):\n    state = _optimizer(model, settings, len(fit_loader), device)\n    best, stale, history, stop = None, 0, [], "max_epochs"\n    for epoch in range(1, settings["max_epochs"] + 1):\n        if history:\n            next_cost = max(row["epoch_seconds"] for row in history)\n            refit_cost = max(row["training_seconds"] for row in history) * full_ratio * epoch + 120\n            if not budget.can_start(next_cost + refit_cost, training=True):\n                stop = "budget_reserving_refit"\n                break\n        try:\n            started = time.monotonic()\n            pilot_context = None\n            if epoch == 1:\n                paths = [local / "pilot_report.json"]\n                if pilot_report_path is not None:\n                    paths.append(pilot_report_path)\n                pilot_context = {"report_paths": paths, "holdout_batches": len(hold_loader), "full_ratio": full_ratio}\n            row = _train_epoch(model, fit_loader, settings, device, state, budget, epoch,\n                               pilot_context=pilot_context)\n            evaluate_started = time.monotonic()\n            predicted = _predict(model, hold_loader, device, budget, training=True)\n            row["evaluation_seconds"] = time.monotonic() - evaluate_started\n            metrics = data.score_predictions(truth, predicted)\n            row["holdout_macro_f1"], row["holdout_metrics"] = metrics["macro_f1"], metrics\n            row["both_macro_and_nausea_improved"] = _selection_key(metrics, reference_metrics)[0]\n            save_started = time.monotonic()\n            if best is None or _selection_key(metrics, reference_metrics) > _selection_key(best["metrics"], reference_metrics):\n                budget.check("save selected epoch")\n                _save_weights(model, local / "selection_weights.pt")\n                np.savez_compressed(local / "holdout_predictions.npz", indices=indices, targets=truth, predictions=predicted)\n                best, stale = {"selected_epochs": epoch, "metrics": metrics}, 0\n            else:\n                stale += 1\n            row["save_seconds"] = time.monotonic() - save_started\n            row["epoch_seconds"] = time.monotonic() - started\n            history.append(row)\n            data.write_json(local / "selection_history.json", history)\n            print(f"{settings[\'candidate_id\']} Training epoch {epoch}: Macro F1={metrics[\'macro_f1\']:.6f}; "\n                  f"{row[\'epoch_seconds\'] / 60:.1f} minutes", flush=True)\n            if stale >= settings["patience"]:\n                stop = "early_stopping"\n                break\n        except BudgetExceeded:\n            stop = "budget_during_epoch"\n            break\n    if best is None:\n        raise BudgetExceeded("No complete Training epoch and holdout evaluation fit the budget")\n    return {**best, "history": history, "stop_reason": stop}\n\n\ndef _manifest(settings, revision, epochs, samples, scope, partial_audit=None):\n    return {"format": "mission3_categorical_v1", "kind": "neural", "version": "ver20",\n            "labels": list(data.TARGET_SYMPTOMS), "decision_rule": "categorical predict/argmax",\n            "model_id": settings["model_id"], "resolved_revision": revision,\n            "selected_epochs": epochs, "fitted_samples": samples, "training_scope": scope,\n            "positive_class_weights": settings["positive_class_weights"], "label_attention": False, "contrastive": False,\n            "trainable_encoder_layers": settings["trainable_encoder_layers"],\n            "expected_encoder_layers": settings["expected_encoder_layers"], "partial_finetuning": partial_audit,\n            "scheduler_horizon_epochs": settings["scheduler_horizon_epochs"],\n            "inference_precision": "CUDA BF16 when supported, otherwise CUDA FP16; CPU FP32",\n            "max_length": settings["max_length"], "overlap": settings["overlap"],\n            "batch_size": settings["batch_size"], "encoder_window_batch": settings["encoder_window_batch"],\n            "assets": "assets", "weights": "selection_weights.pt" if scope == "fit_only" else "final_weights.pt"}\n\n\ndef _device(settings):\n    import torch\n    if not torch.cuda.is_available() and not settings.get("allow_cpu", False):\n        raise RuntimeError("Select a Colab GPU runtime before starting ver20")\n    return torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef _release():\n    import torch\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef train_candidate(texts, targets, fit, holdout, settings, local, budget, pilot_report_path=None,\n                    reference_metrics=None):\n    from .neural import DialogueClassifier, prepare_encoder, seed_everything, configure_partial_finetuning\n    device = _device(settings)\n    seed_everything(settings["seed"])\n    budget.check("prepare candidate encoder", training=True)\n    encoder, tokenizer, revision = prepare_encoder(settings["model_id"], settings["revision"], local / "assets")\n    audit = configure_partial_finetuning(encoder, settings["trainable_encoder_layers"], settings["expected_encoder_layers"])\n    data.write_json(local / "partial_finetuning.json", audit)\n    model = DialogueClassifier(encoder, label_attention=False, contrastive=False)\n    model.encoder_window_batch = settings["encoder_window_batch"]\n    model.to(device)\n    try:\n        budget.check("tokenize candidate", training=True)\n        fit_loader = _loader([texts[i] for i in fit], targets[fit], tokenizer, settings, True)\n        hold_loader = _loader([texts[i] for i in holdout], targets[holdout], tokenizer, settings, False)\n        result = _fit_selection(model, fit_loader, hold_loader, targets[holdout], holdout, settings, device,\n                                local, budget, len(targets) / len(fit), pilot_report_path=pilot_report_path,\n                                reference_metrics=reference_metrics)\n        manifest = _manifest(settings, revision, result["selected_epochs"], len(fit), "fit_only", audit)\n        data.write_json(local / "candidate_model.ckpt", manifest)\n        data.write_json(local / "training_metrics.json", result["metrics"])\n        return manifest, result\n    finally:\n        del model, encoder\n        _release()\n\n\ndef refit_selected(texts, targets, settings, epochs, local, budget):\n    """Only a fully completed fresh refit is eligible for publication."""\n    from .neural import DialogueClassifier, prepare_encoder, seed_everything, configure_partial_finetuning\n    device = _device(settings)\n    seed_everything(settings["seed"])\n    budget.check("prepare fresh refit", training=True)\n    encoder, tokenizer, revision = prepare_encoder(settings["model_id"], settings["revision"], local / "assets")\n    audit = configure_partial_finetuning(encoder, settings["trainable_encoder_layers"], settings["expected_encoder_layers"])\n    data.write_json(local / "partial_finetuning.json", audit)\n    model = DialogueClassifier(encoder, label_attention=False, contrastive=False)\n    model.encoder_window_batch = settings["encoder_window_batch"]\n    model.to(device)\n    try:\n        loader = _loader(texts, targets, tokenizer, settings, True)\n        state = _optimizer(model, settings, len(loader), device)\n        history = []\n        for epoch in range(1, epochs + 1):\n            row = _train_epoch(model, loader, settings, device, state, budget, epoch)\n            history.append(row)\n            data.write_json(local / "final_history.json", history)\n            print(f"Fresh all-Training refit epoch {epoch}/{epochs}: loss={row[\'loss\']:.5f}", flush=True)\n        budget.check("save completed full refit")\n        _save_weights(model, local / "final_weights.pt")\n        probe = _predict(model, _loader(texts[:12], None, tokenizer, settings, False), device, budget)\n        manifest = _manifest(settings, revision, epochs, len(targets), "all_training", audit)\n        data.write_json(local / "final_model.ckpt", manifest)\n        return manifest, probe\n    finally:\n        del model, encoder\n        _release()\n\n\ndef predict_checkpoint(checkpoint, texts, budget=None):\n    import torch\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n    from .neural import DialogueClassifier, configure_partial_finetuning\n    checkpoint = Path(checkpoint)\n    manifest, root = _read(checkpoint), checkpoint.parent\n    if manifest.get("format") != "mission3_categorical_v1" or tuple(manifest["labels"]) != data.TARGET_SYMPTOMS:\n        raise ValueError("Unsupported checkpoint or symptom ordering")\n    if (manifest.get("version") != "ver20" or manifest["kind"] != "neural"\n            or manifest["label_attention"] or manifest["contrastive"]):\n        raise ValueError("ver20 inference requires its single-view classifier")\n    if (manifest.get("model_id") != "klue/roberta-large"\n            or manifest.get("resolved_revision") != "28d911204e9022eda172571ca8cc61eaffd942f7"\n            or manifest.get("expected_encoder_layers") != 24 or manifest.get("trainable_encoder_layers") != 8\n            or manifest.get("scheduler_horizon_epochs") != 3\n            or not isinstance(manifest.get("partial_finetuning"), dict)\n            or manifest.get("decision_rule") != "categorical predict/argmax"):\n        raise ValueError("Checkpoint partial-finetuning provenance is missing or incompatible")\n    if budget:\n        budget.check("load final checkpoint")\n    assets = _safe(root, manifest["assets"])\n    config = AutoConfig.from_pretrained(assets / "encoder", local_files_only=True, trust_remote_code=False)\n    encoder = AutoModel.from_config(config, trust_remote_code=False)\n    audit = configure_partial_finetuning(encoder, manifest["trainable_encoder_layers"], manifest["expected_encoder_layers"])\n    if audit != manifest["partial_finetuning"]:\n        raise ValueError("Saved partial-finetuning audit disagrees with the encoder architecture")\n    tokenizer = AutoTokenizer.from_pretrained(assets / "tokenizer", local_files_only=True, trust_remote_code=False)\n    model = DialogueClassifier(encoder, label_attention=False, contrastive=False)\n    model.encoder_window_batch = manifest["encoder_window_batch"]\n    _load_weights(model, _safe(root, manifest["weights"]))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model.to(device)\n    try:\n        return _predict(model, _loader(texts, None, tokenizer, manifest, False), device, budget)\n    finally:\n        del model, encoder\n        _release()\n\n\ndef inference(checkpoint, label_dir, output, budget=None):\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    predicted = predict_checkpoint(checkpoint, texts, budget)\n    if budget:\n        budget.check("save final predictions")\n    data.write_predictions(paths, predicted, output)\n    return paths, predicted\n\n\ndef _candidate_record(settings, texts, targets, fit, holdout, root, budget, reference_metrics=None):\n    name = settings["candidate_id"]\n    directory = root / "experiments" / name\n    completed = directory / "completed.json"\n    stable = _stable_settings(settings)\n    if completed.is_file():\n        record = _read(completed)\n        if record["settings"] != stable or record["settings_digest"] != _digest(stable) or record["candidate_id"] != name:\n            raise ValueError("Saved candidate configuration changed")\n        if Path(record["directory"]).parts[:2] != ("experiments", name):\n            raise ValueError("Saved candidate directory changed")\n        _verify_hashes(root, record["artifact_hashes"], budget)\n        original = _read(_safe(root, record["directory"] + "/candidate_model.ckpt"))\n        expected = {**original, "weights": record["directory"] + "/" + original["weights"],\n                    "assets": record["directory"] + "/" + original["assets"]}\n        if expected != record["manifest"]:\n            raise ValueError("Saved candidate manifest changed")\n        print("Reusing completed Training candidate:", name, flush=True)\n    else:\n        started, attempt = time.monotonic(), _stamp()\n        local = Path(settings["work_dir"]) / "training" / root.name / name / attempt\n        local.mkdir(parents=True, exist_ok=False)\n        data.write_json(local / "settings.json", stable)\n        manifest, result = train_candidate(texts, targets, fit, holdout, settings, local, budget,\n                                          pilot_report_path=root / "pilot_report.json", reference_metrics=reference_metrics)\n        destination = directory / attempt\n        _publish_tree(local, destination, budget)\n        relative = destination.relative_to(root).as_posix()\n        root_manifest = {**manifest, "weights": relative + "/" + manifest["weights"],\n                         "assets": relative + "/" + manifest["assets"]}\n        record = {"candidate_id": name, "directory": relative, "settings": stable,\n                  "settings_digest": _digest(stable), "manifest": root_manifest,\n                  "metrics": result["metrics"], "history": result["history"],\n                  "stop_reason": result["stop_reason"], "seconds": time.monotonic() - started}\n        record["artifact_hashes"] = _model_hashes(root, root_manifest, (\n            relative + "/holdout_predictions.npz", relative + "/selection_history.json",\n            relative + "/candidate_model.ckpt", relative + "/training_metrics.json", relative + "/settings.json",\n            relative + "/partial_finetuning.json", relative + "/pilot_report.json"))\n        budget.check("commit completed candidate")\n        data.write_json(completed, record)\n    with np.load(_safe(root, record["directory"] + "/holdout_predictions.npz"), allow_pickle=False) as saved:\n        if not np.array_equal(saved["indices"], holdout) or not np.array_equal(saved["targets"], targets[holdout]):\n            raise ValueError("Saved candidate Training targets or split changed")\n        if data.score_predictions(targets[holdout], saved["predictions"]) != record["metrics"]:\n            raise ValueError("Saved candidate scores disagree with discrete predictions")\n    if record["history"] != _read(_safe(root, record["directory"] + "/selection_history.json")):\n        raise ValueError("Saved timing or epoch history changed")\n    return record\n\n\ndef _refit_estimate(record, full_ratio):\n    return (max(row["training_seconds"] for row in record["history"]) * full_ratio\n            * record["manifest"]["selected_epochs"] + 120)\n\n\ndef _write_table(root, records):\n    columns = ["candidate", "selected_epochs", "macro_f1", "nausea_f1", "nausea_precision", "nausea_recall",\n               "seconds", "stop_reason", "positive_class_weights"]\n    temporary = root / "candidate_summary.csv.tmp"\n    with temporary.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns)\n        writer.writeheader()\n        for record in records:\n            nausea = record["metrics"]["per_symptom"][data.TARGET_SYMPTOMS[6]]\n            writer.writerow({"candidate": record["candidate_id"], "selected_epochs": record["manifest"]["selected_epochs"],\n                             "macro_f1": record["metrics"]["macro_f1"], "nausea_f1": nausea["f1"],\n                             "nausea_precision": nausea["precision"], "nausea_recall": nausea["recall"],\n                             "seconds": record["seconds"], "stop_reason": record["stop_reason"],\n                             "positive_class_weights": json.dumps(record["settings"]["positive_class_weights"])})\n    temporary.replace(root / "candidate_summary.csv")\n\n\ndef _final_model(selected, texts, targets, holdout, settings, root, budget, refit_allowed=True):\n    checkpoint, ready = root / "mission3_ver20_model.ckpt", root / "final_model_ready.json"\n    if ready.is_file():\n        saved = _read(ready)\n        if saved["selected_candidate"] != selected["candidate_id"]:\n            raise ValueError("Final model selection changed during resume")\n        _verify_hashes(root, saved["artifact_hashes"], budget)\n        return checkpoint, _read(checkpoint)\n    manifest = dict(selected["manifest"])\n    reason = "budget_before_refit" if refit_allowed else "training_reference_not_jointly_improved"\n    with np.load(_safe(root, selected["directory"] + "/holdout_predictions.npz"), allow_pickle=False) as saved:\n        expected_probe = saved["predictions"][:12].copy()\n    probe_texts = [texts[i] for i in holdout[:len(expected_probe)]]\n    estimate = _refit_estimate(selected, len(targets) / manifest["fitted_samples"])\n    if refit_allowed and budget.can_start(estimate, training=True):\n        attempt = _stamp()\n        local = Path(settings["work_dir"]) / "training" / root.name / "full_refit" / attempt\n        local.mkdir(parents=True, exist_ok=False)\n        final_settings = {**settings, **selected["settings"]}\n        data.write_json(local / "settings.json", _stable_settings(final_settings))\n        try:\n            new_manifest, new_probe = refit_selected(texts, targets, final_settings,\n                                                     manifest["selected_epochs"], local, budget)\n        except BudgetExceeded:\n            reason = "budget_during_refit"\n            print("Refit did not finish within its training budget; retaining the completed fit-only candidate.", flush=True)\n        else:\n            destination = root / "final_model" / attempt\n            _publish_tree(local, destination, budget)\n            relative = destination.relative_to(root).as_posix()\n            manifest = {**new_manifest, "weights": relative + "/" + new_manifest["weights"],\n                        "assets": relative + "/" + new_manifest["assets"]}\n            expected_probe, probe_texts = new_probe, texts[:len(new_probe)]\n            data.write_json(root / "final_history.json", _read(local / "final_history.json"))\n            reason = None\n    elif refit_allowed:\n        print("Full refit does not fit the remaining budget; using the completed fit-only model.", flush=True)\n    else:\n        print("Skipping full refit: Training Macro F1 and nausea F1 did not both improve. "\n              "The saved fit-only candidate is experimental; retain the prior submission.", flush=True)\n    manifest.update(selected_candidate=selected["candidate_id"], refit_skipped_reason=reason)\n    data.write_json(checkpoint, manifest)\n    actual = predict_checkpoint(checkpoint, probe_texts, budget)\n    if not np.array_equal(actual, expected_probe):\n        raise ValueError("Reloaded final model differs from its saved selection or in-memory predictions")\n    hashes = _model_hashes(root, manifest, (checkpoint.name,))\n    budget.check("commit verified final model")\n    data.write_json(ready, {"selected_candidate": selected["candidate_id"], "reloaded_probe_samples": len(expected_probe),\n                           "artifact_hashes": hashes})\n    return checkpoint, manifest\n\n\ndef _environment():\n    import torch\n    from importlib.metadata import version, PackageNotFoundError\n    packages = {}\n    for name in ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification", "tokenizers"):\n        try:\n            packages[name] = version(name)\n        except PackageNotFoundError:\n            packages[name] = "not installed"\n    return {"packages": packages, "cuda_available": torch.cuda.is_available(),\n            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}\n\n\ndef run(settings, runtime_sources=None):\n    from .reference import verify_training_reference, compare_reference, reference_summary\n    started = time.monotonic()\n    settings = validate_settings(settings)\n    deadline = settings.get("deadline_monotonic", started + settings["time_budget_minutes"] * 60)\n    if deadline > started + settings["time_budget_minutes"] * 60 + 1:\n        raise ValueError("Supplied deadline exceeds the allowed execution budget")\n    budget = Budget(deadline, finalization_reserve_s=settings["finalization_reserve_seconds"],\n                    safety_factor=settings["budget_safety_factor"])\n    sources = runtime_sources or {p.name: p.read_text(encoding="utf-8") for p in Path(__file__).parent.iterdir()\n                                 if p.is_file() and p.suffix in (".py", ".md", ".txt", ".json")}\n    sources = {key: value for key, value in sources.items() if not key.startswith("test_") and key != "build_notebook.py"}\n    output = Path(settings["output_dir"]).expanduser().resolve()\n    resume = settings.get("resume_run", "")\n    root = Path(resume).expanduser().resolve() if resume else output / _stamp()\n    if root == output or not root.is_relative_to(output):\n        raise ValueError("resume_run must be a child run directory inside output_dir")\n    if resume and not (root / "run_identity.json").is_file():\n        raise ValueError("resume_run is not initialized")\n    if not resume:\n        root.mkdir(parents=True, exist_ok=False)\n    print("Result run:", root, flush=True)\n    writable_status = not bool(resume)\n    records, skipped = [], []\n    try:\n        budget.check("prepare inputs")\n        _device(settings)\n        train_dir, validation_dir = prepare_data(settings["input_zip"], settings["work_dir"],\n                                                settings.get("train_dir", ""), settings.get("validation_dir", ""))\n        paths, texts, targets = data.load_dataset(train_dir, include_targets=True)\n        fit, holdout = data.make_split(texts, targets, settings["holdout_size"], settings["seed"])\n        training_digest = dataset_digest(texts, targets)\n        reference = verify_training_reference(training_digest, targets, fit, holdout,\n                                              split_seed=settings["seed"], holdout_size=settings["holdout_size"])\n        identity = {"version": "ver20", "training_digest": training_digest,\n                    "source_digest": _digest({k: v for k, v in sources.items() if not k.endswith(".md")}),\n                    "settings_digest": _digest(_stable_settings(settings)), "candidates": CANDIDATES,\n                    "model_selection": SELECTION_POLICY}\n        # JSON normalizes tuples, so identity is stored and compared in that canonical representation.\n        identity = json.loads(json.dumps(identity))\n        if resume:\n            if _read(root / "run_identity.json") != identity:\n                raise ValueError("Resume source, settings, or Training data changed")\n            with np.load(root / "training_split.npz", allow_pickle=False) as saved:\n                if not np.array_equal(saved["fit"], fit) or not np.array_equal(saved["holdout"], holdout):\n                    raise ValueError("Resume Training split changed")\n        else:\n            data.write_json(root / "run_identity.json", identity)\n            data.write_json(root / "settings.json", {k: v for k, v in settings.items() if k != "deadline_monotonic"})\n            np.savez_compressed(root / "training_split.npz", fit=fit, holdout=holdout)\n            submission = root / "submission"\n            submission.mkdir()\n            for name, source in sources.items():\n                if Path(name).name != name:\n                    raise ValueError("Runtime sources must have flat filenames")\n                (submission / name).write_bytes(source.encode("utf-8"))\n        if (root / "run_complete.json").is_file():\n            _verify_hashes(root, _read(root / "run_complete.json")["artifact_hashes"], budget)\n            print("Run already complete; its saved final artifacts were verified.", flush=True)\n            return root\n        writable_status = True\n        data.write_json(root / "training_reference_verified.json", reference_summary(reference))\n        data.write_json(root / "runtime.json", {"status": "running", "seconds": time.monotonic() - started,\n                        "message": "Only run_complete.json confirms completed final artifacts."})\n        data.write_json(root / ("environment_" + _stamp() + ".json"), _environment())\n        data.write_json(root / "experiment_plan.json", {"candidates": CANDIDATES, "ce_views": 1,\n                        "selection": SELECTION_POLICY, "max_epochs": settings["max_epochs"],\n                        "training_reference": "ver19 saved fit-only predictions; no baseline retraining",\n                        "encoder_layers": {"total": settings["expected_encoder_layers"],\n                                           "trainable": settings["trainable_encoder_layers"]},\n                        "time_budget_minutes": settings["time_budget_minutes"],\n                        "finalization_reserve_seconds": settings["finalization_reserve_seconds"],\n                        "budget_fallback": "completed fit-only candidate; never incomplete refit"})\n        final_already_ready = (root / "final_model_ready.json").is_file()\n        for name, weights in CANDIDATES:\n            completed = root / "experiments" / name / "completed.json"\n            if not completed.is_file() and final_already_ready:\n                skipped.append({"candidate": name, "reason": "final_model_already_selected"})\n                continue\n            if not completed.is_file() and records:\n                previous = records[0]\n                minimum_trial = max(row["epoch_seconds"] for row in previous["history"]) * min(3, settings["max_epochs"]) + 120\n                current_best = max(records, key=lambda row: row["metrics"]["macro_f1"])\n                estimate = minimum_trial + _refit_estimate(current_best, len(targets) / len(fit))\n                if not budget.can_start(estimate, training=True):\n                    skipped.append({"candidate": name, "reason": "budget_reserving_refit"})\n                    print("Skipping additional candidate to preserve finalization/refit time:", name, flush=True)\n                    continue\n            config = {**settings, "candidate_id": name, "positive_class_weights": weights}\n            try:\n                records.append(_candidate_record(config, texts, targets, fit, holdout, root, budget,\n                                                  reference_metrics=reference["metrics"]))\n            except BudgetExceeded:\n                skipped.append({"candidate": name, "reason": "budget_no_completed_epoch"})\n                if not records:\n                    raise\n                break\n            _write_table(root, records)\n        if not records:\n            raise BudgetExceeded("No completed candidate is available")\n        selected = max(records, key=lambda row: row["metrics"]["macro_f1"])\n        data.write_json(root / "final_selection.json", {"candidate_id": selected["candidate_id"],\n                        "metrics": selected["metrics"], "selection_policy": SELECTION_POLICY,\n                        "selected_epochs": selected["manifest"]["selected_epochs"], "skipped_candidates": skipped})\n        data.write_json(root / "training_metrics.json", selected["metrics"])\n        saved = _safe(root, selected["directory"] + "/holdout_predictions.npz")\n        shutil.copy2(saved, root / "holdout_predictions.npz")\n        with np.load(saved, allow_pickle=False) as values:\n            comparison = compare_reference(reference, values["predictions"], targets, holdout)\n        data.write_json(root / "training_reference_comparison.json", comparison)\n        qualified = comparison["both_macro_and_nausea_improved"]\n        print("Training comparison with ver19: both Macro F1 and nausea F1 improved =", qualified, flush=True)\n        if not qualified:\n            print("This is an experimental candidate, not a verified improvement. Existing results remain unchanged.", flush=True)\n        checkpoint, manifest = _final_model(selected, texts, targets, holdout, settings, root, budget,\n                                             refit_allowed=qualified)\n        hashes = _read(root / "final_model_ready.json")["artifact_hashes"]\n        for name in ("training_reference_comparison.json", "training_reference_verified.json",\n                     "holdout_predictions.npz", "training_metrics.json", "final_selection.json", "pilot_report.json"):\n            hashes[name] = _hash(root / name)\n        if settings.get("run_validation", True):\n            predicted_paths, predicted = inference(checkpoint, validation_dir, root / "mission3.csv", budget)\n            budget.check("score final Validation CSV")\n            scored_paths, _, truth = data.load_dataset(validation_dir, include_targets=True)\n            if [p.name for p in scored_paths] != [p.name for p in predicted_paths]:\n                raise ValueError("Validation row order changed")\n            metrics = data.score_predictions(truth, predicted)\n            data.write_json(root / "validation_metrics.json", metrics)\n            print(f"Final Validation Macro F1={metrics[\'macro_f1\']:.6f}", flush=True)\n            for name in ("mission3.csv", "validation_metrics.json"):\n                hashes[name] = _hash(root / name)\n        limited = bool(skipped) or manifest["training_scope"] != "all_training" or any(r["stop_reason"].startswith("budget") for r in records)\n        budget.check("commit final run")\n        status = "complete_not_improved" if not qualified else "complete_budget_limited" if limited else "complete"\n        data.write_json(root / "runtime.json", {"status": status,\n                        "seconds": time.monotonic() - started, "seconds_scope": "worker invocation; notebook deadline includes installation",\n                        "candidate_count": len(records), "selected_candidate": selected["candidate_id"],\n                        "both_macro_and_nausea_improved_on_training": qualified,\n                        "training_reference_version": "ver19", "baseline_retrained": False,\n                        "trainable_encoder_layers": manifest["trainable_encoder_layers"],\n                        "training_scope": manifest["training_scope"], "fitted_samples": manifest["fitted_samples"],\n                        "refit_skipped_reason": manifest["refit_skipped_reason"], "skipped_candidates": skipped,\n                        "budget_minutes_remaining": max(0., budget.remaining() / 60)})\n        data.write_json(root / "run_complete.json", {"artifact_hashes": hashes})\n    except BudgetExceeded:\n        if writable_status:\n            data.write_json(root / "runtime.json", {"status": "budget_exhausted", "seconds": time.monotonic() - started,\n                            "candidate_count": len(records), "skipped_candidates": skipped,\n                            "message": "No newly completed run; existing completed candidate artifacts are preserved."})\n        print("Time budget exhausted. Completed artifacts remain in the result directory.", flush=True)\n    except BaseException as error:\n        if writable_status:\n            data.write_json(root / "runtime.json", {"status": "interrupted" if isinstance(error, (KeyboardInterrupt, SystemExit)) else "failed",\n                            "seconds": time.monotonic() - started, "error_type": type(error).__name__})\n        raise\n    return root\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--audio_dir", required=True)\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args(argv)\n    inference(args.ckpt_path, args.label_dir, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""Official one-command inference entry point for a portable result folder."""\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nif __package__:\n    from .runtime import main\nelse:\n    root = Path(__file__).resolve().parent\n    name = "mission3_categorical_submission"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    from importlib import import_module\n    main = import_module(name + ".runtime").main\n\nif __name__ == "__main__":\n    main()\n', 'reference.py': '"""Immutable, fit-only historical Training holdout comparison for ver20.\n\nThe embedded reference contains discrete predictions and aggregate provenance,\nnot text, identifiers, Validation labels, probabilities, or a trained model.\nIt is a descriptive benchmark; repeated use of this holdout limits generality.\n"""\nfrom __future__ import annotations\n\nimport base64\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zlib\n\nimport numpy as np\n\nfrom . import data\n\nREFERENCE_PATH = Path(__file__).with_name("training_reference.json")\n# Set from the independently rebuilt, verified historical artifact.\nREFERENCE_FILE_SHA256 = "6979dfc0d92e7441ebd50eb1fc531b9cd11f91eab3bb9aae3912784b6e9a5581"\n\n\ndef array_digest(values, dtype):\n    """Hash shape and canonical bytes, independent of host integer byte order."""\n    values = np.ascontiguousarray(values, dtype=np.dtype(dtype))\n    digest = hashlib.sha256(json.dumps(list(values.shape)).encode("ascii") + b"\\0")\n    digest.update(values.tobytes())\n    return digest.hexdigest()\n\n\ndef _indices(values, total, name):\n    values = np.asarray(values)\n    if (values.ndim != 1 or values.dtype.kind not in "iu" or not len(values)\n            or (values < 0).any() or (values >= total).any()\n            or len(np.unique(values)) != len(values)):\n        raise ValueError(f"Historical reference: invalid {name} indices")\n    return values.astype("<i8")\n\n\ndef _require_equal(actual, expected, name):\n    if actual != expected:\n        raise ValueError(f"Historical reference mismatch: {name}")\n\n\ndef _metrics_match(actual, expected):\n    if isinstance(expected, dict):\n        return (isinstance(actual, dict) and set(actual) == set(expected)\n                and all(_metrics_match(actual[key], value) for key, value in expected.items()))\n    if isinstance(expected, (int, float)) and not isinstance(expected, bool):\n        return (isinstance(actual, (int, float)) and np.isfinite(actual)\n                and np.isfinite(expected) and abs(actual - expected) <= 1e-12)\n    return actual == expected\n\n\ndef _decode_predictions(payload, rows):\n    _require_equal(payload.get("encoding"), "zlib+base64/int8-c-order", "prediction encoding")\n    _require_equal(payload.get("shape"), [rows, len(data.TARGET_SYMPTOMS)], "prediction shape")\n    try:\n        compressed = base64.b64decode(payload["data"], validate=True)\n        decoder = zlib.decompressobj()\n        expected_size = rows * len(data.TARGET_SYMPTOMS)\n        raw = decoder.decompress(compressed, expected_size + 1)\n        if (len(raw) != expected_size or not decoder.eof or decoder.unused_data\n                or decoder.unconsumed_tail):\n            raise ValueError("Historical reference: invalid compressed prediction size")\n        predictions = np.frombuffer(raw, dtype=np.int8).reshape(rows, len(data.TARGET_SYMPTOMS)).copy()\n    except (KeyError, TypeError, ValueError, zlib.error) as error:\n        raise ValueError("Historical reference: corrupt prediction payload") from error\n    predictions = data._binary(predictions, rows)\n    _require_equal(array_digest(predictions, "i1"), payload.get("sha256"), "prediction checksum")\n    return predictions\n\n\ndef verify_training_reference(training_digest_value, targets, fit, holdout, *, path=None,\n                              split_seed=42, holdout_size=0.2):\n    """Fail closed unless the historical rows, targets, split, and payload match.\n\n    ``training_digest_value`` must be the runtime\'s digest of the current,\n    canonically sorted Training texts and targets. No Validation is accessed.\n    The returned ``predictions`` ndarray is for comparison, not JSON output.\n    """\n    path = REFERENCE_PATH if path is None else Path(path)\n    raw = path.read_bytes()\n    _require_equal(hashlib.sha256(raw).hexdigest(), REFERENCE_FILE_SHA256, "reference file checksum")\n    reference = json.loads(raw.decode("utf-8"))\n    _require_equal(reference.get("format"), "mission3_training_reference_v1", "format")\n    _require_equal(reference.get("labels"), list(data.TARGET_SYMPTOMS), "label order")\n    _require_equal(reference.get("training_scope"), "fit_only", "historical training scope")\n    _require_equal(reference.get("decision_rule"), "categorical predict/argmax", "decision rule")\n    _require_equal(training_digest_value, reference.get("training_digest"), "Training data digest")\n    _require_equal(split_seed, reference.get("split_seed"), "split seed")\n    _require_equal(holdout_size, reference.get("holdout_size"), "holdout proportion")\n    targets = data._binary(targets, reference["training_samples"], "targets")\n    fit = _indices(fit, len(targets), "fit")\n    holdout = _indices(holdout, len(targets), "holdout")\n    if (len(fit) + len(holdout) != len(targets)\n            or len(np.union1d(fit, holdout)) != len(targets)):\n        raise ValueError("Historical reference: fit/holdout must partition Training")\n    _require_equal(len(fit), reference["fit_samples"], "fit sample count")\n    _require_equal(len(holdout), reference["holdout_samples"], "holdout sample count")\n    hashes = reference["array_hashes"]\n    _require_equal(array_digest(fit, "<i8"), hashes["fit_indices"], "fit order")\n    _require_equal(array_digest(holdout, "<i8"), hashes["holdout_indices"], "holdout order")\n    _require_equal(array_digest(targets, "i1"), hashes["training_targets"], "Training targets")\n    _require_equal(array_digest(targets[holdout], "i1"), hashes["holdout_targets"], "holdout targets")\n    predictions = _decode_predictions(reference.pop("predictions_payload"), len(holdout))\n    _require_equal(array_digest(predictions, "i1"), hashes["predictions"], "reference prediction hash")\n    metrics = data.score_predictions(targets[holdout], predictions)\n    if not _metrics_match(metrics, reference["metrics"]):\n        raise ValueError("Historical reference mismatch: recomputed metrics")\n    reference["predictions"] = predictions\n    reference["verified_reference_file_sha256"] = hashlib.sha256(raw).hexdigest()\n    return reference\n\n\ndef reference_summary(reference):\n    """Return verified aggregate metadata suitable for manifests and JSON."""\n    return {key: value for key, value in reference.items() if key != "predictions"}\n\n\ndef compare_reference(reference, new_predictions, targets, holdout):\n    """Compare raw argmax decisions on the exact same Training holdout rows."""\n    targets = data._binary(targets, reference["training_samples"], "targets")\n    holdout = _indices(holdout, len(targets), "holdout")\n    hashes = reference["array_hashes"]\n    _require_equal(array_digest(holdout, "<i8"), hashes["holdout_indices"], "comparison holdout order")\n    _require_equal(array_digest(targets, "i1"), hashes["training_targets"], "comparison Training targets")\n    truth = targets[holdout]\n    old = data._binary(reference["predictions"], len(holdout))\n    _require_equal(array_digest(old, "i1"), hashes["predictions"], "comparison reference predictions")\n    new = data._binary(new_predictions, len(holdout))\n    old_metrics = data.score_predictions(truth, old)\n    if not _metrics_match(old_metrics, reference["metrics"]):\n        raise ValueError("Historical reference mismatch: comparison reference metrics")\n    new_metrics = data.score_predictions(truth, new)\n    nausea = "오심"\n    macro_delta = new_metrics["macro_f1"] - old_metrics["macro_f1"]\n    nausea_delta = new_metrics["per_symptom"][nausea]["f1"] - old_metrics["per_symptom"][nausea]["f1"]\n    per_symptom = {}\n    for index, label in enumerate(data.TARGET_SYMPTOMS):\n        y, before, after = truth[:, index].astype(bool), old[:, index].astype(bool), new[:, index].astype(bool)\n        per_symptom[label] = {\n            "delta": {key: new_metrics["per_symptom"][label][key] - old_metrics["per_symptom"][label][key]\n                      for key in ("f1", "precision", "recall", "false_positive", "false_negative")},\n            "paired": {\n                "fn_to_tp": int((y & ~before & after).sum()),\n                "tp_to_fn": int((y & before & ~after).sum()),\n                "fp_to_tn": int((~y & before & ~after).sum()),\n                "tn_to_fp": int((~y & ~before & after).sum()),\n                "persistent_fn": int((y & ~before & ~after).sum()),\n                "persistent_fp": int((~y & before & after).sum()),\n            },\n        }\n    return {\n        "available": True, "reference_version": reference["version"],\n        "reference_candidate": reference["candidate_id"], "comparison_scope": "Training internal holdout only",\n        "decision_rule": reference["decision_rule"], "samples": len(holdout),\n        "reference_metrics": old_metrics, "new_metrics": new_metrics,\n        "macro_f1_delta": macro_delta, "nausea_f1_delta": nausea_delta,\n        "both_macro_and_nausea_improved": bool(macro_delta > 0 and nausea_delta > 0),\n        "status": ("both_improved" if macro_delta > 0 and nausea_delta > 0 else\n                   "macro_only_improved" if macro_delta > 0 else\n                   "nausea_only_improved" if nausea_delta > 0 else "neither_improved"),\n        "per_symptom": per_symptom,\n        "limitations": reference["limitations"],\n    }\n', 'training_reference.json': '{\n  "format": "mission3_training_reference_v1",\n  "version": "ver19",\n  "candidate_id": "nausea_gce",\n  "run_id": "20261008T072419Z_07dc5385",\n  "training_scope": "fit_only",\n  "decision_rule": "categorical predict/argmax",\n  "labels": [\n    "고열",\n    "구토",\n    "두통",\n    "복통",\n    "어지러움",\n    "열상",\n    "오심",\n    "전신쇠약",\n    "호흡곤란"\n  ],\n  "split_seed": 42,\n  "holdout_size": 0.2,\n  "training_samples": 29200,\n  "fit_samples": 23360,\n  "holdout_samples": 5840,\n  "training_digest": "1d0c54e5f27b507663f5690fce7c2fd350b20b33f4bafe2aabbfd4370bf30525",\n  "selected_epochs": 3,\n  "historical_model": {\n    "model_id": "klue/roberta-base",\n    "resolved_revision": "02f94ba5e3fcb7e2a58a390b8639b0fac974a8da",\n    "loss_mode": "nausea_ce_gce"\n  },\n  "array_hashes": {\n    "fit_indices": "08b71a25fd798963eec81d98775112a6355d88be110c37e33d5adf37685cb437",\n    "holdout_indices": "20f8fcb447a5ca4de0434da94680da48a58b46f6c2c2e03139b60f8738e5b0ed",\n    "training_targets": "416fe9b7e8cf31471124b89a41fcc14255d12bc8441970b4d6203957fb286074",\n    "holdout_targets": "e43046eb56f91c77a261820c1ccf5151f5adf1911fbe45ff41f953e6cc307d7e",\n    "predictions": "b470a8060e202fe6749c35e7779d2354530821039762e576a41e9fefd550101b"\n  },\n  "metrics": {\n    "macro_f1": 0.6412755833526849,\n    "micro_f1": 0.6520991272217621,\n    "samples": 5840,\n    "per_symptom": {\n      "고열": {\n        "f1": 0.6606129398410897,\n        "precision": 0.8027586206896552,\n        "recall": 0.5612343297974928,\n        "support": 1037,\n        "false_positive": 143,\n        "false_negative": 455\n      },\n      "구토": {\n        "f1": 0.5701058201058201,\n        "precision": 0.6962843295638126,\n        "recall": 0.4826427771556551,\n        "support": 893,\n        "false_positive": 188,\n        "false_negative": 462\n      },\n      "두통": {\n        "f1": 0.531021897810219,\n        "precision": 0.5650485436893203,\n        "recall": 0.5008605851979346,\n        "support": 581,\n        "false_positive": 224,\n        "false_negative": 290\n      },\n      "복통": {\n        "f1": 0.8100759696121551,\n        "precision": 0.8831734960767219,\n        "recall": 0.7481536189069424,\n        "support": 1354,\n        "false_positive": 134,\n        "false_negative": 341\n      },\n      "어지러움": {\n        "f1": 0.6765531062124248,\n        "precision": 0.6872964169381107,\n        "recall": 0.6661404893449092,\n        "support": 1267,\n        "false_positive": 384,\n        "false_negative": 423\n      },\n      "열상": {\n        "f1": 0.8862094951017332,\n        "precision": 0.9274447949526814,\n        "recall": 0.8484848484848485,\n        "support": 693,\n        "false_positive": 46,\n        "false_negative": 105\n      },\n      "오심": {\n        "f1": 0.3708523096942095,\n        "precision": 0.32796317606444186,\n        "recall": 0.42664670658682635,\n        "support": 668,\n        "false_positive": 584,\n        "false_negative": 383\n      },\n      "전신쇠약": {\n        "f1": 0.5856079404466501,\n        "precision": 0.6190975865687304,\n        "recall": 0.5555555555555556,\n        "support": 1062,\n        "false_positive": 363,\n        "false_negative": 472\n      },\n      "호흡곤란": {\n        "f1": 0.6804407713498623,\n        "precision": 0.7406296851574213,\n        "recall": 0.6292993630573248,\n        "support": 785,\n        "false_positive": 173,\n        "false_negative": 291\n      }\n    }\n  },\n  "predictions_payload": {\n    "encoding": "zlib+base64/int8-c-order",\n    "shape": [\n      5840,\n      9\n    ],\n    "sha256": "b470a8060e202fe6749c35e7779d2354530821039762e576a41e9fefd550101b",\n    "data": "eNqdnduW3UYIRNn//9N5cDwjddcFOQ9eSXxmjtQXKAooZv78w8+fzPufv3/D8694/PnnX94f4Odnfn/k/pnnR9/f+PjS69vN7/n97Mz9q1Ff+nq84wfTu76/hvef9/ucH1bPIj76fH25aI8XwK0F1zOar+PYJ/F5923XD1ybde2vWqDz4B3fIM+J2nDxwqjFRr6b+ArkISe83fkW4dXuS3OtPOXBxLEhbDFh5f0+6e9636HnJUBfq2RtyskkrDOwMA3HM3Pt75wGTZrB628Ry3N8hu2eWisCz3vPa6dRtkLYh0mbMvJukS+puvv3+XzsznUV0b9Wb0qyZpwLfnwSfxmjoUZthHcB7qwSvJNwed6uoy7c5HtKMzv2bCLuyHW7o2lKX0i4qfjrrx7tcIuM3Dy1KFiHxpTjZ4zR7y1lttfyvDg8D8UsDpg62zhrTLALW3wirZ/wuQIfGXiJ/o7so5BPcDyPspD4hRqEQ0YsNcVXWpj0tI2HT0lXmDcWQhlzjv9NBCLq8MDthnAnAG3QiMdBuMZiLgg77j7NzycVKL0/bOwDDQkHc2CstT204K4at/3H3shxUMuaU8bfi/GAgYQheG6BvPlEJGSgNhp6PC8IBpM9Vl6fjRBoZA+BPcrEl1NoPb3g+1KVZ77sOSygwdMDyWAEbQAvOIf7Xkp85IkE/t9ahw9x/tSfeYJnE4+BjREUfGaBgz5AFHGWV7DQhNgng6MP1tyRb7z5RHNBw6vEcO9pEzJQuNx68ssEPoEG5Y2Bw4SB2LjTMhXCenoGrCCiY+WQBv+OPOmrQ3QKpDNJswmUX0iLNX5X/n4XCKEvEcphEBGZmFNfgrUt+uIW3smiqw8+QNua9b84ZxYPpEUFq9iW/M0TcCgTyAFyoOqjpev7IUf4FgEWBqZFbTSc9SDTFDxv72645Ozz8ZQXnm3G76A7CCiMHRwi0Z0R/Yj0G5bKwp1NzwuFE4ANMSyVZA8wC7qDJyF25msiC7ogrxrAMtFEAPPubDhEFAltG4CPpeMx1EcMrpnKkTrv+QNCshVFxbyY+3ibixqAeu+Au+t4h4KLIufYNI2NkPecZo5xbEnCTvdFTjDyaTN5cg8xTocVufdPqZ/gqUucctndRvZSOOEGBoh2i8KF4b6BbMNTEjnmPd9UHvHueOBkgCIb2joc3OAiJaN7eth13gGJvrDOtdDXZ1IJbgdEQIKe6ithQAw77l3O1FCKBstB4X7Zy5KgTjXBweIRdcTUAuNZeE7NAKmMIYmEpwXyIouPj0qW3KAnHt8+dJOKCxDl5kbTs5QIr3H8HuRQQssp0R8+rZNinCvxTYuRiKebmZrfqDiz8xK+ZsYtAuOrCDDXkJSZzfR1yCBsciDxnln22sTEntT1eMaRD5gDlv4njaKsrBfq7kSTdYBwi3tNCmtlrf7mZWZzMsfmAdOdJ9tFTadTzBCT8RevvKh4eMOCxHOfXlwTyI5jnF5kOLvtljcOQ4gEy79KaCKqCPQJp8XpPmeDKhUK3HthmWU8dVkfFjwmxbMHH8eqdGpRjyQRUXEBM4uYiAV5LII7EV+7qA3DdHeWuJgv6eNu9oCY3J6Ya/9SOWM9EZmMzkaPSpsvyXfNpX4rs0mcrObwQ6g4zbJlkqHyTNRHj+ePRAnjbpwoprJFa2ELtvU2BEtC3y25UCR7a5KkHqIdi5oTq+fLE8rd3tZb+PfeLVBMgYtGqA6IRnXNT1kM+yRObAFYnITFgwWuPcWaLMxpDz1cqWRKPaAtyt4wOlNOfKltntrUyjp3ZWM3DE+kWf0We15Okd1V1qmxm+R14ZU6mRrBqxV7fcRg1B0T4TeTtAps2AxVltmy7HRDTUxqT0uR0aLjueqlcUmPc3NL7Ud5SsQuGydBvbioLiF8AVj9vaHVhw0dYStOCyNEoRpCY0zEkEgbRz425zlQFTvEnqmvRxQEy2ZKEfGXOkVVcrEp/IZPhpfYd2NI8Ety779PtrFJEdS83sMOpGo3mVgR1phwWgwXxl2M7eMTMrRWFQCpkqkRDvt8DCG3sUBNXPWjp6fBE9ENqok7iA3JYqPT5F5cTqIglPsyC//Xcy7uQbYZlmAcYjkrOa5ovBidx07uP4T2HLwa7jLxofCyUZN+lVL/V8jQkxdUo27Ric4sQEthWkjQKEG+M6rV5KSuCsvncLVUtsAq0BH28pCOMwGDOj58QZgcnpHiNIlU3oRCA6pPtX1kzFTeuOOFXhCrQYNpfERfB8y9I0RJkdTV1RXYhEplKgRKx/gvpqTVL+zEDiwVFhGrt+ArJWrLx/ROYAuvqf/Rqyxi+N8qMQy90HPrfEj95HzKkSawLZM+jqMfCwdFbayikj7edyw6hDb8mLILveAjBOkpUcOdCGsAyKkTRMIM/ZNY7/V4+FgH9IJHLChzZzyzkzLWtV1lQlX5FUHHS+TlZWon3MsINAyQQ9wvVabxArCLczXV8DUrRA6lD1yHKah6BNnaRnv6hOD6zYnK4Qn9MzSaSRa3mSDTH8yb9tm1W6f6zF0QIVlxZLPKgkZGsm7ZxBE8WrYE7j1ZuP6EVPHmYlOgIvMDqtevlbWGEK8wgY2ATRXepKiQ8Bh07zYLL1canXzrR0Et4E9EDrt8ofgNWVwDV/nNBBpa2VOLmrqGkGczHdnrb5YpwqwNd7iMAWY9g8Qd4dhQc14k3qaWQTrQd4VcVyphwYkjodSiaKSz40csoxqScPyr7hG77Q3fLS6qvAolpIPsiY+oMiwLqZgphvChaJsvDczpw2xSijIqwysbBrbw2Fx6H7S2xngDwtn7vaoFjXU6ETGblgQVJ27yH+9ywHU5yi2O5euiG6WckFWC06zl+AzLWvsn9pqNBb2dL1jK4noELYpMUv9iTehSCPzY0RWL/73UTMzCTEwHS12IjRargcSSxiTuM4ugNm5u4wECNTxaou289DS7Kh1kpX+vR+qiBCYlyLEX7Co/fT1r4UJ2Sgj63Lj+u8/f9AsqPrTjBXUC93vwW3c0Nj4sfxAt9vwu2YDkviR5nlflcNHH2VVlNlhUW7wIXFetcJ0RC73bUvIwWfyW13Nhr64JNAacFh2tylo2mdGo6WEjoFrp3aS3wnMjrpOLhVUGtKjSTNRqOI0nKz7x+c7YyhqLrBaiKJap/mf1IX+esxyEc8baXxyqmTQEqLWQg9N0FAoFHDhAHolSPsCZxodvC6b0rdmW+ZEUxRZimbrhMbBq5Lx+CKC4zS+VWS9F+6ygOeUonu6R6zrTz/PELFXNc/nvItzkZm0WkvKbKPJ9Ya3FXQV6FyJYinvH/n570LNtI6JlG0u5DutZVFJVnhaVT/4xRzXOHdvC2ROt48uKSOm80xFqVvxCEKwIilz8vBS/yr+nR6FJKEkfTNaeaNI5DmKkGfthnSQrYL8lJI0lJBKFm8An52zRuiJR6FDO6GnJPZ+mCfH7FXWVYkUanW49WvMqTO2MDKLUMSwhcoy9UeX1S9FFfLWRp63XJh4Mz7zVADdycyxCbL1UpGjiQ0MiG90/76W9Zvssco0Lg6ehyAIy4TMt2Etp23YWnBixJplaNF6OYooWWLSuBAOV6ib5QHU2+n4TaCSWYDu9oSRM0oCPMD+qutZVvSjJ27FpjssdqR3bBtHgLvPtUUEdRqDwaopPSj0p9mEIMax2Glb9Ljk7xAQNVs7ASY8eDk9zx8TTiSPGg/dHOzuF1Us1gbGvrTepz5Kg82MqMLHGgT6Eb3FHpksfp0yN7P7RWxmSQZaHLM+IRruRhBc2Ck3GU3nCueqVN+p5LgtMlhSbNPPS30W7HP7GeUGFhSeyfAIumUthQTvF/wtd8OJOWeaXDiasCug0mYvEpST5hWg08SBKO4cjHRIqBFaXb8x0rVnK6DXh4m84BTHkyzPtXoIgQlSfP2252tNbEaP+Fo+zZYFmFSYSL6Njg8srt35PPXLIlURb2nCVyqzSLrG+lIXwNoGEo+ExCTvbwIdArS733ZeEEwMxH9MTFTmKjFVscxA87zcllslHiI/x13k82TBrdsId9HixcAXU4l0TCvraVDnVKFNQy5Ex/dVsxa15J6uj9Tlz3a8QnlJ2FEqDuzsAIn3uVO59oTXO6Lm75hwuB5GRrFbkavopt49hQGDcdyofnrT2bkRkcxOYNJWvU1iPVsHwG7NssB1dS4yznHGmyX1NvOyhUa3HMkCp8NfzCU61fj93qRYFGl8SJnssCG0KzmxqVTtm8pXzClM9T4Cy0US5kB6lYsoF19kfG4WQKUliIkedJyDj783OX1ZC5HTQTKhJZ9Mx6eBSrAnGVgDWuHyMxEdMD2wA5+xtYcrMUgpilyk/4/POoT+j0mh93ECyoixYteY266jWvN2YM09W4KHGVD1xgfESSNZW729yZLUxglkAGWY5yLFMh+R2660SqrrRpL2ZfA/nwd5QZ5ecPrPokokEk698Je0edbTuW1biK7vC5Ir5HDaNy02WsXoNfp9E4SbOylOo1Y9sFRU8ygloNkHaYIgqo0jJC6cAgcuqLzDGRnp0IqWJpZ1ruIZNWZjRhMtWV521p1a01unZNpAtbp5YRBgJJM9wkEgimrwbJoW0GAO7q4+nDF+m1OA5a9O7IXtpVxMjD7tZLNNeFZV0Zd7ym9eK0WLDcvaowQCuxiKAKMIpYflXSeGZrGumWze397nulNneRRSppFUnUhFtHHkrH83TKq1fYD99rzuy1YChY8QPmyGt3ZunroFt1O9Q4OmcLK9uQDJy8pq7RjTCBKujeQj5FoeoC0dZSc2wK5V8x7mQnJ9M3OwU6zRPGC8+weALJod6DYqwDdnd0MbCjivPLu+1EGhsFnutvebiCWKtkb+wBAy2mdxXAtue0yzcxGr2k3yJ0s2Eo30WmLVNFSGNL9TJ8bGGmGafVyJhyEPRENoJGy6lyiJs4fTVhcKls7b1IBHoJeOF5cyIlIWh+dxZzBqmTRZLEC5dqyIy5uQy+DSpuWXz8tjTEF1tFF22Q6ZDSQb/NK07VlOQ6PRaa72TONsZnxAam6XPKWn8zLvFSJysdNSkXdpTIlnApVlueH4zPDr1izQJuM7iVz5cpVOCqhKbU8aGQfN3h3IHPRGBu1YYcJWYnkXlp8uRrwb2aZQlyvRzYaQ3SJtpjg1VZmVJNm5NQ8cq+WE6zgjBxGSXtCXBbc5lI2TObZxCrWvu2MVf+YyU8D0enn6ZNJpvh2kDK6ROSavHsykAl6QhJ9ondYQ1Niw7J1Jbt6beWDOTXoeT0G+gjRyp3dZaq89tIFuNRA6+dqUc/x5nE7LOaWLItIE5Lgfd+vK6srTy/hGz4WKvS3AAWWo7DZFrlEex4YK4Zp0GT8o5kSRZaqHIppjoc20olLRLNgmqxEef9jzMg5bBAlFx6zp2+LhzVWMWSC7HVNTkL5ILOCPxe6R6DOSZLLkvossO/8czgd86/EL4Rsq+Zqo7FJJRnW9XYIxX5cYnVWbQKg516oyZSkz3OqV741gFy4vZybWDxjb7V41TJkyzSoQA6BhkKqSe1PHUC1CnjFRehcm3zbqGuWgnQUvb/6xYHfTDjfOlME5ks9ejmUprir+g+OxMuBCu2jkkFYKqCe91xTm/R6FvE0qPIxilybNYrlBDbo4wjjGZ8GH+8czH8Sepge735OOIvEBE0AlLrwyTyJn8pgp9pgkQKzat1JjYPgVic7LlCahjqbEKWjjKAkelxnpxOiO9mmETCqWxIYxDvt/41VkNDbq8Tv21FAT7hVdfYJLtv/25sNjwMHNyEej8mAIaNWxTYmfte9WUkwzGWPE90p7KuEDt+0ZO2Ory2a4dCgb2/PbLU/vZKq2Aviihszpej+egkQxx/k7tR3v9j51+QhLZ4E2v1Vb9bNB0ZqZFQKTutqmz1AO8NomVNAas8BP3GtI9+nSxl00/ZcM59/wfX/UkgLwILcMA+sJtDKNKDiI5TBvHgF0ALTsoKdlYZpC5FIdJja7Il+ECbQzjrREblesWQw8t0fYtu9kg6KLFwuoat2qqrvzvG5Oy9cWFRKkpz3WNuCphn20icWR396emPBZ91Ub0gET4a+RgaeEzIsjHgWpJcDJNNes8fr5szgTcsQ9mEHBgxqi5gAV90MYJ/dXj6OMziLOSO553PT6xuw7XcUlhX87q4Smz5v3IobXwO8YBEVKHid1sPeFOKlcCKHIjRemJzP0bi5x2puWjcudKWpYFzDRXwg7LIy4CfYACsUI2ZIlrX4y7M8Gx7mgx1momEUjTFDm6xP2sivpwdP0rEiozW99vutNLWQ1ZdHgeBUfDCCv87B5fThFm+W0d/dggyW8Gzj0FKFvZ+NSStpw7Eey8mfOIkEKlY/1R5Q2NtNxi4vvo5BGxsv45JoFSSEWUPA/uaByJ2ls7W2MPM8sbHbiykEZJMtcThEyNH+/tveoxPJUCHtdtrNjsJNwXLjbT+2GGSaGL4D7um74RFtCENVUuKnJTE+5q7/zKiz7xJtNtvr1ObZyk8FSvd5So2jGgbhF4ElPceQ73U+0w8K9V06H0pFUi44SSqdIPInTFE7bKOVKA/Z1T4IsuehJbwCpIyqEGmdreuP5bTvc/iQMcvg=="\n  },\n  "provenance": {\n    "source_run_training_digest": "1d0c54e5f27b507663f5690fce7c2fd350b20b33f4bafe2aabbfd4370bf30525",\n    "source_run_source_digest": "4a4064efe4a5c81fa52fad8e000157ab0f23d75854503d3972d7e57f3beeadb4",\n    "source_hashes": {\n      "experiments/nausea_gce/20261008T080049Z_74b9b8c0/holdout_predictions.npz": "84e2d1f158cf83d6000055d5ead377a5c479bb034fe054037fdea208b91888b1",\n      "experiments/nausea_gce/20261008T080049Z_74b9b8c0/training_metrics.json": "6e70346a77b296613a6dfe3927b3c65443bb38c3ad26d344539bb178e273695b",\n      "experiments/nausea_gce/completed.json": "959c5f5029e198ad5c276e4084dacc57b9e73e5817ca5702a02f9a3524fb9944",\n      "training_split.npz": "524a04326803a3eb41696b322a9a76e20f8afcfdbcccd37ebe089ebc6f77e264",\n      "run_identity.json": "507e1030d9749fa1ea39e5b68f7966d76bfee44d88901b377517e43e5f612f51",\n      "final_selection.json": "cd3de6fc7eb388ff5a359a11bcd43d50690fbbb8cf43934fed3fdeb17ad0524b"\n    },\n    "source_selection_weights_sha256": "7cf8eb27e79dae667ee146f05ccc25a50f5048a0b201110f52a86ac86131bd11",\n    "recreated_training_split_matches": true,\n    "original_training_targets_match": true,\n    "validation_accessed": false\n  },\n  "limitations": [\n    "This Training holdout has been used repeatedly across previous versions; it is not a fresh test set.",\n    "Historical ver19 and new ver20 differ in architecture and training recipe; this is a model comparison, not an isolated causal experiment.",\n    "The historical predictions came from fit-only training, never the full-Training final model.",\n    "Improvement here does not guarantee higher official Validation or hidden-test F1."\n  ]\n}\n', 'run_notebook.py': '"""Launch the bounded ver20 experiment in a fresh interpreter after installation."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\n\nSOURCE_FILES = (\n    "__init__.py", "data.py", "bootstrap.py", "neural.py", "budget.py", "runtime.py",\n    "inference.py", "reference.py", "training_reference.json", "run_notebook.py", "requirements.txt", "README.md",\n)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result_file", required=True)\n    args = parser.parse_args(argv)\n    root = Path(__file__).resolve().parent\n    name = "mission3_ver20_execution"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    runtime = importlib.import_module(name + ".runtime")\n    settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n    sources = {name: (root / name).read_text(encoding="utf-8") for name in SOURCE_FILES}\n    result = runtime.run(settings, runtime_sources=sources)\n    Path(args.result_file).write_text(json.dumps(str(result)), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\n', 'README.md': "# Mission 3 ver20\n\n## One candidate: KLUE-RoBERTa-large with partial fine-tuning\n\n`large_partial_ce` starts from `klue/roberta-large`, pinned to revision `28d911204e9022eda172571ca8cc61eaffd942f7`. The official model has 24 encoder layers and hidden size 1024. Embeddings and the first 16 encoder layers are frozen; the last 8 layers and nine two-class symptom heads train. The base classification and weighted cross-entropy recipe follow ver15. There is no GCE, utterance-evidence branch, label attention, contrastive loss, threshold search, or post-hoc score adjustment.\n\nThe objective is to improve shared Korean context representation while limiting training cost. Eight trainable layers are an experimental budget choice, not an established optimum. Large-model F1 and complete GPU runtime have not been measured locally, and improvement is not guaranteed. Sources: [official model](https://huggingface.co/klue/roberta-large), [pinned revision](https://huggingface.co/klue/roberta-large/commit/28d911204e9022eda172571ca8cc61eaffd942f7), [official configuration](https://huggingface.co/klue/roberta-large/blob/28d911204e9022eda172571ca8cc61eaffd942f7/config.json).\n\nInputs remain only `utterances[].text`. All supplied utterances, overflow windows, samples, and original labels remain. Windows use maximum length 512 with overlap 64. Batch size 4 and gradient accumulation 4 give 16 conversations per full optimizer step. Encoder windows are processed in chunks of 4. Encoder learning rate is 2e-5, head learning rate 1e-4, and both the maximum training length and scheduler horizon are three epochs. Positive weights in `[고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란]` order are `[1.4, 1.8, 2.0, 1.2, 1.3, 1.0, 3.0, 1.5, 1.3]`. Predictions are raw two-class argmax decisions for each symptom.\n\n## Evaluation and the embedded Training reference\n\nThe existing grouped Training split uses seed 42 and holdout size 0.2. Actual indices are saved to `training_split.npz`. Epoch selection uses only Training holdout results: prefer epochs whose Macro F1 and nausea F1 both exceed the verified ver19 reference, then choose the highest Macro F1 among them. If no epoch passes both conditions, retain the highest-Macro-F1 epoch as a diagnostic result and skip full refitting. This prevents a later high-Macro but lower-nausea epoch from discarding an earlier jointly improved epoch. There is one newly trained candidate and no baseline retraining.\n\n`training_reference.json` embeds prior Training-holdout binary predictions and provenance hashes, without patient text or sample IDs. `reference.py` verifies the current Training data, labels, and split before comparing to the embedded reference. `training_reference_comparison.json` reports historical metrics, per-label changes, and nausea error transitions. Its primary reference is ver19. Joint improvement requires both overall Macro F1 and nausea F1 to increase. Failure of either condition skips full refitting to save time: the completed fit-only model and experimental predictions remain, with status `complete_not_improved` and `refit_skipped_reason: training_reference_not_jointly_improved`. They are not presented as a newly recommended submission. This gate does not load an old trained model or replace the new experiment with an older checkpoint. Previous outputs remain separate.\n\nThis reused holdout is exploratory and is not an untouched independent test. Reference information is not a model input, loss term, sampler, threshold, or label correction. Official Validation is evaluated once after final model selection and is not used for training, epoch selection, or reference comparison.\n\n## Run the standalone notebook\n\nOpen `mission3_ver20.ipynb` in Colab, select a GPU, and run all cells. Connect the shared Drive account containing the existing `mission3_ver5_inputs.zip`. Keep `train_dir` and `validation_dir` empty for ZIP discovery. Runtime code and reference data are embedded in the notebook; previous results do not need to be uploaded. Existing pinned dependencies are unchanged. Results use a new unique directory under `DCC_AIdeas/results/mission3/ver20`.\n\nThe shared 180-minute deadline starts after Drive login and before dependency installation. Installation has a 20-minute cap. A 25-minute reserve protects final evaluation, saving, and publication; measured phase estimates use a 1.25 safety factor. The pilot uses one initial optimizer step as warmup followed by 10 measured optimizer steps when enough batches exist (44 microbatches at the defaults). Warmup and measured steps are all part of epoch 1, not additional or repeated training. The pilot estimates feasible epochs and records device and peak CUDA memory. Later epochs consider the time needed for a fresh all-Training refit. Timing estimates are provisional and can differ from actual throughput or Drive write speed.\n\nIf a completed candidate fits but a full refit does not, the completed candidate may be published with explicit `training_scope: fit_only` and the reason. If no complete candidate can be produced, the run stops as incomplete. Interrupted epochs and partial refits never replace complete models. The deadline limits the job budget but cannot guarantee full Training refitting and finalization on every GPU.\n\n`AUTO_RELEASE_GPU=True` requests Colab runtime release after displaying results, on errors, or at the shared deadline. Drive artifacts remain; `/content` scratch can disappear. A failed release request prints the manual Runtime > Disconnect and delete runtime instruction. Real service behavior must be checked in Colab.\n\n## Results and resuming\n\nKeep the full run directory and executed notebook. Review `pilot_report.json`, `candidate_summary.csv`, `training_reference_comparison.json`, `final_selection.json`, `training_metrics.json`, `validation_metrics.json`, and `runtime.json`. The model manifest is `mission3_ver20_model.ckpt`; it references weights and assets in the same run directory. Check its actual `training_scope`, `fitted_samples`, and `selected_epochs`. The portable inference entry point is `submission/inference.py` and prediction CSV header is `label file name,symptom`.\n\nOnly `run_complete.json` together with status `complete`, `complete_budget_limited`, or `complete_not_improved` indicates completed artifacts. `complete_not_improved` explicitly means the joint Training improvement requirement failed and full refitting was skipped; its CSV is the fit-only model's experimental output. Missing final artifacts or completion marker means incomplete work. Abrupt Colab reclamation may leave the previous status unchanged. Completion describes successful generation of this experiment's artifacts, not improved F1 or permission to replace an earlier submission.\n\nSet `resume_run` only to a compatible ver20 directory. Code, settings, data, split, embedded reference, and artifact hashes are checked before reuse. A complete compatible run is verified and returned without retraining. Partial epochs are not resumed. Each explicit rerun starts a new time budget and consumes additional Colab resources. Older notebooks, results, and dependencies remain unchanged.\n"}
try:
    runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver20_"))
    for name, source in RUNTIME_SOURCES.items():
        (runtime_dir / name).write_text(source, encoding="utf-8", newline="\n")
    budget_spec = importlib.util.spec_from_file_location("mission3_ver20_budget_supervisor", runtime_dir / "budget.py")
    budget_module = importlib.util.module_from_spec(budget_spec)
    sys.modules[budget_spec.name] = budget_module
    budget_spec.loader.exec_module(budget_module)
    print("실행 코드 준비 완료")
except BaseException:
    release_gpu()
    raise


In [ ]:
# @title 기존 프로젝트와 같은 버전의 패키지 준비 (최대 20분, 전체 예산에 포함)
package_deadline = min(DEADLINE_MONOTONIC, time.monotonic() + 20 * 60)
try:
    budget_module.run_supervised(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(runtime_dir / "requirements.txt")],
        deadline_monotonic=package_deadline, shutdown_margin_s=10,
    )
    print("패키지 준비 완료 · 남은 작업 시간:", round(max(0, DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")
except BaseException:
    release_gpu()
    raise


In [ ]:
# @title 후보 학습 → 최종 평가 · 결과 표시 · GPU 반환
import csv
import json
try:
    settings_file = runtime_dir / "notebook_settings.json"
    result_file = runtime_dir / "notebook_result.json"
    settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    # A fresh interpreter uses installed versions; the deadline includes all work.
    budget_module.run_supervised(
        [sys.executable, "-u", str(runtime_dir / "run_notebook.py"),
         "--settings", str(settings_file), "--result_file", str(result_file)],
        deadline_monotonic=DEADLINE_MONOTONIC, shutdown_margin_s=10,
    )
    result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))
    print("결과 폴더:", result_dir)
    summary = result_dir / "candidate_summary.csv"
    if summary.is_file():
        print("Training 내부 비교:")
        with summary.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                print(row["candidate"], "Macro F1 =", round(float(row["macro_f1"]), 6),
                      "오심 F1 =", round(float(row["nausea_f1"]), 6),
                      "선택 epoch =", row["selected_epochs"], "종료 이유 =", row["stop_reason"])
    pilot_path = result_dir / "pilot_report.json"
    if pilot_path.is_file():
        pilot = json.loads(pilot_path.read_text(encoding="utf-8-sig"))
        print("초기 GPU 속도 측정:", pilot.get("device", "확인 필요"),
              "· 갱신 횟수:", pilot.get("observed_optimizer_steps"),
              "· epoch 예상 분:", round(pilot["estimated_epoch_seconds"] / 60, 1),
              "· 계획:", pilot.get("decision"))
    comparison_path = result_dir / "training_reference_comparison.json"
    if comparison_path.is_file():
        comparison = json.loads(comparison_path.read_text(encoding="utf-8-sig"))
        if comparison["available"]:
            nausea = comparison["per_symptom"]["오심"]["paired"]
            print("동일 Training 내부 평가 기준:", comparison["reference_version"],
                  "· 전체 F1 변화:", round(comparison["macro_f1_delta"], 6),
                  "· 오심 F1 변화:", round(comparison["nausea_f1_delta"], 6))
            print("오심 누락 회복:", nausea["fn_to_tp"], "· 새 누락:", nausea["tp_to_fn"],
                  "· 오탐 제거:", nausea["fp_to_tn"], "· 새 오탐:", nausea["tn_to_fp"])
            if comparison["both_macro_and_nausea_improved"]:
                print("Training 내부 평가에서 전체 F1과 오심 F1이 모두 개선됐습니다.")
            else:
                print("전체 F1과 오심 F1의 동시 개선 조건을 충족하지 못했습니다:", comparison["status"])
            print("이전 모델과 결과는 유지됩니다. 새 실험 결과가 자동으로 기존 제출 모델을 교체하지 않습니다.")
        else:
            print("기존 Training 결과 비교 불가:", comparison.get("reason", "기록 확인 필요"))
    for filename in ["training_metrics.json", "validation_metrics.json"]:
        path = result_dir / filename
        if path.is_file():
            report = json.loads(path.read_text(encoding="utf-8-sig"))
            print(filename, "Macro F1 =", round(report["macro_f1"], 6))
            for symptom, values in report["per_symptom"].items():
                print(symptom, "F1 =", round(values["f1"], 4),
                      "P =", round(values["precision"], 4), "R =", round(values["recall"], 4))
    manifest_path = result_dir / "mission3_ver20_model.ckpt"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        print("선택된 후보:", manifest.get("selected_candidate", "모델 정보 확인"))
        print("저장 모델 학습 범위:", manifest["training_scope"],
              "· 학습 표본:", manifest["fitted_samples"], "· epoch:", manifest["selected_epochs"])
        print("모델:", manifest.get("model_id", SETTINGS["model_id"]),
              "· 학습 인코더 층:", SETTINGS["trainable_encoder_layers"], "/", SETTINGS["expected_encoder_layers"])
        if manifest.get("refit_skipped_reason"):
            print("전체 재학습 생략 이유:", manifest["refit_skipped_reason"])
    status_path = result_dir / "runtime.json"
    status = {}
    if status_path.is_file():
        status = json.loads(status_path.read_text(encoding="utf-8-sig"))
        print("실행 상태:", status.get("status", "기록 확인 필요"))
        if status.get("skipped_candidates"):
            print("생략한 후보:", status["skipped_candidates"])
    print("이번 노트북 소요 시간:", round((time.monotonic() - BUDGET_STARTED_MONOTONIC) / 60, 1), "분")
    print("선택 근거:", result_dir / "final_selection.json")
    print("학습 범위가 기록된 모델 정보:", manifest_path)
    complete = ((result_dir / "run_complete.json").is_file()
                and status.get("status") in {"complete", "complete_budget_limited", "complete_not_improved"})
    if complete:
        if status.get("status") == "complete_not_improved":
            print("실험 완료 · 동시 개선 미달로 전체 재학습을 생략했습니다.")
            print("새 추천 제출 모델이 아닙니다. 기존 제출 결과를 유지하고 비교 보고서를 확인하세요.")
            print("내부 fit 학습 모델의 실험용 예측:", result_dir / "mission3.csv")
        else:
            print("최종 작업 완료 · 예측:", result_dir / "mission3.csv")
    else:
        print("미완료 실행:", status.get("status", "상태 기록 없음"),
              "· 저장된 후보 결과와 로그는 결과 폴더에 남아 있습니다. 최종 제출 결과로 사용하지 마세요.")
finally:
    release_gpu()


## 결과 확인 · 중단 후 재개

기본 실행은 `ver20/실행시각_고유번호/`에 저장됩니다. 이전 버전의 노트북과 결과는 유지합니다.
`pilot_report.json`, `candidate_summary.csv`, `training_reference_comparison.json`, `final_selection.json`,
`training_metrics.json`, `validation_metrics.json`, `runtime.json`을 확인하세요.
모델 `.ckpt`의 `training_scope`가 전체 Training 학습인지 내부 fit 학습인지를 알려 줍니다.
시간 때문에 전체 재학습을 생략한 경우에도 실제 완료 범위를 명시합니다.
`complete_not_improved`는 동시 개선 미달로 전체 재학습을 생략한 실험 완료 상태입니다.
이 상태의 예측은 내부 fit 학습 모델의 실험 결과이며 새 추천 제출 모델이 아닙니다.
`budget_exhausted` 또는 최종 예측·평가 파일 누락은 완료된 제출 결과가 아닙니다.
강제 종료·Colab 회수 시 마지막 `runtime.json` 상태가 최종 종료 이유를 반영하지 못할 수 있습니다.
`run_complete.json`은 최종 완료 표시이므로 이 파일이 없는 실행을 완료로 간주하지 마세요.

재개하려면 `resume_run`에 이전 실행 폴더의 전체 Drive 경로를 넣고 다시 실행합니다.
재개 가능 여부는 코드·설정·데이터 일치 여부로 확인하며, 완료된 결과만 재사용합니다.
이미 `complete`, `complete_budget_limited`, `complete_not_improved`인 실행을 재개하면 저장된 결과를 검증하고 반환하며,
추가 학습하지 않습니다. 새 실험을 하려면 `resume_run`을 비우세요.
중단된 epoch 중간부터 이어 학습하지 않습니다. **재실행에는 새 시간 예산과 추가 Colab 리소스가 듭니다.**

모델 `.ckpt`는 가중치·assets의 상대 경로를 담은 manifest입니다.
**실행 폴더 전체와 실행 로그가 있는 노트북을 보관**하세요. 제출용 추론 코드는 `submission/inference.py`입니다.
예측 CSV의 헤더는 `label file name,symptom`입니다.
기본 설정은 결과 표시 후 또는 제한 시간에 Colab `runtime.unassign()`으로 GPU 반환을 요청합니다.
`AUTO_RELEASE_GPU`를 끄거나 자동 반환이 실패하면 **런타임 → 연결 해제 및 런타임 삭제**를 직접 실행하세요.
자동 반환 기능의 실제 서비스 동작은 Colab에서 확인해야 합니다.
